# E17 — SGLang silent-accuracy tests (Qwen3.8-27B on 4x L4), engine only

PLAN_E17 rev 2 (Experiment 14 `Bug ang fix/`, 2026-10-26; three GPT audits merged). One session, every server TP4 with
the production numerics (RedHat INT4 W4A16, FP8 e4m3 KV, fp16 SSM, extra_buffer_lazy, overlap, decode CUDA graphs).

- **T1** (#43567, the stale lazy Mamba checkpoint): upstream `test_mamba_cache_kl.py` Class 1 ported. A 4-slot,
  1-running server; warm `[44]*128`, `[42]*128`; the 255-token prompt scored before and after a 32-token seed;
  full-vocabulary KL. Arms: interval 128 (hook on, hook off), interval 256 (+ a 511-token prompt), the forced-fail env,
  DFlash W6 at interval 64 (prompts 246 / 249 / 252). Then the same arms with the #43567 backport. Patch-safety cases
  (EOS, abort, max_new, 5x reuse, cold) with a slot-leak check.
- **L6**: upstream Class 2 (no_buffer + MTP), a known stale-restore path in 0.5.21 (#43387): it shows that the harness
  can detect a stale restore.
- **T2** (only if T1 shows the bug, patched then unpatched): production flags, 16 streams, 4 shared ~30k prompts; failure
  and stale-donation rates; warm vs cold probes.
- **T3b**: target-only vs DFlash, radix cache off, serial and 16 concurrent, greedy 2,048 tokens. **T3a**: fp16 vs fp32
  SSM on 8k-48k prefixes.

**Default profile `E17SHORT` (user 2026-10-26: at most 60 min):** only L1 / L7 (ordinary decode + the patch-safety cases)
and L5 / L9 (DFlash W6), the decisive #43567 pairs. The full plan above (`Q27X_PROFILE=E17`, 170 min) runs later, and
only if the bug shows.

Instrumentation: `e17_hook` as `sitecustomize` on PYTHONPATH of the server only; it wraps four SGLang methods in place
and never edits the source. Verdict ceiling: ENGINE_CORRECTNESS_VALIDATED. Report `e14/e17_report.md`, bundle
`Q27X_E17_BUNDLE.zip`. Commit only (the rerun writes the cell-0 fallback).

**Inputs:** competition data; RedHat INT4, DFlash2 bf16, FP8 repack (the tokenizer); the SGLang 0.5.21 wheelhouse; the
vLLM wheelhouse (the tokenizer pins); the payload dataset (E14 modules).


In [ ]:
# ==================================================================== CELL 0 — mode + fallback submission (stdlib only)
import os, sys, json, time, random, hashlib, re, shutil, subprocess, threading, traceback, zipfile, math
from pathlib import Path
from datetime import datetime, timezone
import concurrent.futures as cf

NOTEBOOK_T0 = time.perf_counter()
# S1 (CLOCK): Kaggle's 12 h counts from kernel start; R5's cell 0 began 7 s later. Anchor the clock at the kernel
# process start (capped at 10 min in case the value is odd).
try:
    import psutil as _ps
    _since_kernel_start = max(0.0, min(600.0, time.time() - _ps.Process(os.getpid()).create_time()))
except Exception:
    _since_kernel_start = 0.0
NOTEBOOK_T0 -= _since_kernel_start
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RERUN = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true", "yes"}
ARC_ROOT = Path(os.environ.get("Q27X_ARC_ROOT", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"))
if not ARC_ROOT.exists():
    for _alt in ("/kaggle/input/arc-prize-2026-arc-agi-2",):
        if Path(_alt).exists():
            ARC_ROOT = Path(_alt)
WORKING = Path(os.environ.get("Q27X_WORKING", "/kaggle/working")); WORKING.mkdir(parents=True, exist_ok=True)
OUT = WORKING / "q27x"; OUT.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = WORKING / "submission.json"

def _load_json(p):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

test_challenges = _load_json(ARC_ROOT / "arc-agi_test_challenges.json")

def write_json_atomic(path, obj):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f)
    os.replace(tmp, path)

def identity_submission(challenges):
    return {tid: [{"attempt_1": q["input"], "attempt_2": q["input"]} for q in task["test"]]
            for tid, task in challenges.items()}

write_json_atomic(SUBMISSION_PATH, identity_submission(test_challenges))
print("MODE:", "COMPETITION RERUN (hidden test)" if RERUN else "COMMIT / INTERACTIVE (evaluation)",
      "| fallback submission.json:", len(test_challenges), "tasks | RUN_ID", RUN_ID,
      "| clock anchored", round(_since_kernel_start, 1), "s before cell 0")


In [ ]:
# ==================================================================== CELL 1 — configuration
PROFILE = "RERUN_ALL" if RERUN else os.environ.get("Q27X_PROFILE", "E17SHORT").strip().upper()
PROFILE_TABLE = {
    # commit run: 12 public evaluation tasks, scored; also measures throughput for the 12-hour projection
    "EVAL12":    {"source": "eval", "n_tasks": 12, "time_budget_sec": 3.0 * 3600, "admit_frac": 0.80, "drain_frac": 0.90},
    # S1 quick commit: 4 eval tasks that closed fastest in R5 iteration 1 -> a valid version in about 1 h, then submit
    "S1COMMIT4": {"source": "eval", "task_list": ["1818057f", "2ba387bc", "bf45cf4b", "58490d8a"],
                  "time_budget_sec": 90 * 60, "admit_frac": 0.80, "drain_frac": 0.90},
    # R5 EVAL120: full evaluation run (120 tasks / 172 inputs) on the hidden-run clock (RERUN_ALL margins)
    "EVAL120":   {"source": "eval", "n_tasks": 120, "time_budget_sec": 6.0 * 3600,   # E15P rev 2: = S60 6H
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
    # E9/E10: PROXY48 (EDA LB-mirror) + the 8 EVAL12 tasks not in it = 56 tasks / 72 test inputs
    "EVAL14":    {"source": "eval", "task_list": ["135a2760", "136b0064", "20a9e565", "21897d95", "247ef758", "2b83f449", "3dc255db", "4c3d4a41", "4c416de3", "4c7dc4dd", "62593bfd", "64efde09", "65b59efc", "6e4f6532"],
                  "time_budget_sec": 2.0 * 3600, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
    # E12 step 3.5: GPU smoke, 1 fast task (2ba387bc: 5,355 thinking tokens in E10 ORDER2), 2 branches, 15 min solving
    "SMOKE1":    {"source": "eval", "task_list": ["2ba387bc"], "time_budget_sec": 50 * 60, "admit_frac": 0.90,
                  "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60,
                  "admit_margin_sec": 2 * 60, "solve_window_sec": 15 * 60},
    # E12 step 4: the E11-05 EVAL30 tasks (same seed sample), 4 h
    "EVAL30":    {"source": "eval", "n_tasks": 30, "time_budget_sec": 4.0 * 3600,
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
    # E12 rev 4 (user 2026-10-20): EVAL6 = 6 tasks that S49/R5/DIRECT solved and E12 EVAL30 did not (accuracy test)
    "EVAL6":     {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd"],
                  "time_budget_sec": 125 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 90 * 60},   # audits: the 100k cap is soft; 12 x ~120k = 1.44 M at 368 tok/s = 65 min + prefill
    # E12 rev 5 (user 2026-10-21): the same 6 tasks with every validated fix (A/B against rev 4 EVAL6: 3/6, 4/8)
    "EVAL6R5":   {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd"],
                  "time_budget_sec": 125 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 90 * 60},
    # E12 rev 5 (user 2026-10-21): EVAL6R5 + 6 informative EVAL30 tasks: 4 Duck rev 1 solved (247ef758 Duck-only on
    # query 1 and first-exact-wrong; bf45cf4b 9.5k; f931b4a8 38.8k; 65b59efc 67k) + 2 mid ones it missed (9385bd28 and
    # cbebaa4b, ledger solve rate 0.25). 24 branches / 6 live; 150 min solving, 185 min budget
    "EVAL12R5":  {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 185 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 150 * 60},
    # E12 rev 5: the E12 EVAL30 tasks (same seed sample) with every fix (A/B against rev 1 EVAL30 7/48 and Poetiq R5 16/48).
    # Capacity: rev 4 EVAL6 measured about 10.4 min per task at 6 live -> 30 tasks about 5.2 h of solving
    "EVAL30R5":  {"source": "eval", "n_tasks": 30, "time_budget_sec": 6.0 * 3600, "admit_frac": 0.90, "drain_frac": 0.94,
                  "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 330 * 60},
    # E12 rev 6 (user 2026-10-21): the EVAL12R5 tasks with rev 6 (A/B: rev 5 xhigh EVAL6R5 5/8, low EVAL12R5 10/19)
    "EVAL12R6":  {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 185 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 150 * 60},
    "EVAL30R6":  {"source": "eval", "n_tasks": 30, "time_budget_sec": 6.0 * 3600, "admit_frac": 0.90, "drain_frac": 0.94,
                  "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 330 * 60},
    # gate 4b (user 2026-10-21): the EVAL12R6 tasks at the hidden run's compute share. CELL 6 sets the solve window to
    # n_tasks x (hidden solve seconds / 240), the hidden solve end taken from RERUN_ALL and the start measured in this run
    "EVAL12R6G4B": {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                    "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                    "time_budget_sec": 80 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                    "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                    "hidden_equiv": True},
    # R6-X1 (user 2026-10-21): target-only vs DFlash2 parity on 4 tasks (no solving): DFlash2 twice (control), then the
    # same server without --speculative-*, the same requests at T 0, one at a time, thinking on and off
    "PARITYR6":  {"source": "eval", "task_list": ["136b0064", "bf45cf4b", "247ef758", "b5ca7ac4"],
                  "time_budget_sec": 75 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "parity": True},
    # E14-A (user 2026-10-23: one notebook; plan Notes/13): engine measurements (A1), a 25-minute Duck run with the
    # adopted levers on the G4B tasks (raw reasoning on), more engine arms (A2). Every phase has its own deadline;
    # whatever does not fit before the hard stop is skipped and reported, never squeezed.
    "E14A":      {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 200 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 25 * 60, "e14": True},
    # E14S (Notes/30, 31; user 2026-10-24: a short commit): a valid version + a GPU smoke of the submission arm (DFlash
    # width 6 + CUDA graphs 1..13) at 12 streams, no measurements. 36 min, not 30: R2 was READY at ~12.5 min (install
    # 3 + staging 2 + first launch 502 s) and the SGLang give-up is admit - 5 min (30 min -> 13 min, 36 min -> 19 min)
    "E14S":      {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 36 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 8 * 60},
    # E14S EVAL120 gate 4b (user 2026-10-24): every evaluation task (120 / 172 inputs) with the hidden run's compute per
    # task: CELL 6 sets the window to 120 x (hidden solve seconds / 240) = about 5.8 h; the coverage of the hidden run
    # (stage-1 starts, branch 1 starts, pass1 / pass2) on eval difficulty. 6.5 h budget = start + window + margins
    # E15 (Notes/35): engine arms (no E14A measurements: E14_DO_MEASURE stays False, so the first launch = the submission
    # arm W6 + graphs13 = the reference), then a 15-min Duck run on the G4B tasks with the adopted arm. 180 min budget (about 15 launches)
    "E15":       {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 180 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 15 * 60, "e15": True},
    # E15B commit (user 2026-10-25, Exp 15 Notes/08): 20 eval tasks with BOTH attempts finishing = the 16 G4B tasks + the
    # first 4 by id of the tasks solved in 1-2 of the 3 EVAL120 G4B runs (2026-10-24). 40 attempts on 16 streams = 3 waves;
    # rev 3f / 3g: waves 16 x a1, then 4 x a1 + 12 x a2, then the last 4 x a2 (4 running = faster). 45k + 30k at 22 tok/s:
    # worst ~ 83 min -> window 92, budget 115; 40k + 35k: worst ~ 74 min -> window 85, budget 110 (READY took 12-13 min;
    # the solve deadline = hard stop - 180 s still clamps it)
    "E15B":      {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b",
                                                  "2ba387bc", "1818057f", "45a5af55", "9aaea919",
                                                  "16de56c4", "1ae2feb7", "221dfab4", "269e22fb"],
                  "time_budget_sec": 128 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 105 * 60},
    # E15B EVAL120 gate 4b: every evaluation task at the hidden run's compute per task (as E14SEVAL120G4B)
    "E15BEVAL120G4B": {"source": "eval", "n_tasks": 120, "time_budget_sec": 6.5 * 3600, "admit_frac": 0.90,
                       "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                       "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "hidden_equiv": True},
    # rev 3i (user 2026-10-25): all 120 eval tasks, exactly a 6-h notebook; the solve window = the hidden run's per-task
    # share (120 x (hidden solve end - start) / 240 ~ 5.8 h), clamped by this run's hard stop - 180 s (~5.7 h)
    "E15BEVAL120R3I": {"source": "eval", "n_tasks": 120, "time_budget_sec": 6.0 * 3600, "admit_frac": 0.90,
                       "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                       "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "hidden_equiv": True},
    # E15P (user 2026-10-25: Poetiq on the Duck engine): all 120 evaluation tasks, a 6-h notebook like the E11-05 S60
    # EVAL120 6H run (3af4660a) and the Duck R3K EVAL120 runs; the solve runs to the drain (12 min before the hard stop)
    "E15PEVAL120": {"source": "eval", "n_tasks": 120, "time_budget_sec": 6.0 * 3600, "admit_frac": 0.90,
                    "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                    "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
    # E17 (PLAN_E17 rev 2, 2026-10-26): engine-only silent-accuracy tests; the 12 G4B tasks only feed the task cells;
    # 170 min = first launch ~9 min + ~12 relaunches ~1.5 min + the T1 / T2 / T3 blocks (~85 min) + reserve
    "E17":       {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd", "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 170 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "e17": True},
    # E17SHORT (user 2026-10-26, the default): the first session, <= 60 min, only the decisive #43567 pairs L1 / L7 and
    # L5 / L9; the full E17 plan runs later only if the bug shows
    "E17SHORT":  {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd", "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 60 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "e17": True,
                  "e17_short": True},
    "E14SEVAL120G4B": {"source": "eval", "n_tasks": 120, "time_budget_sec": 6.5 * 3600, "admit_frac": 0.90,
                       "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                       "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "hidden_equiv": True},
    "PROXY56":   {"source": "eval", "task_list": ["0934a4d8", "20a9e565", "4c7dc4dd", "89565ca0", "136b0064", "291dc1e1", "16b78196", "271d71e2", "28a6681f", "2b83f449", "2c181942", "31f7f899", "35ab12c3", "3dc255db", "409aa875", "4c416de3", "64efde09", "6e453dd6", "7491f3cf", "7b0280bc", "7b80bb43", "7c66cb00", "1818057f", "332f06d7", "53fb4810", "7666fa5d", "da515329", "8f3a5a89", "db695cfb", "97d7923e", "de809cff", "c7f57c3e", "cb2d8a2c", "58490d8a", "dbff022c", "5961cc34", "80a900e0", "269e22fb", "8b9c3697", "e376de54", "b99e7126", "c4d067a0", "135a2760", "4a21e3da", "71e489b6", "800d221b", "dd6b8c4b", "36a08778", "21897d95", "247ef758", "4c3d4a41", "6e4f6532", "88bcf3b4", "898e7135", "8b7bacbf", "b6f77b65"], "time_budget_sec": 6.0 * 3600,
                  "admit_frac": 0.80, "drain_frac": 0.90},
    # hidden test: every task; Kaggle's limit is 12 h including install + model load
    # v1.4.3: stop 10 min before 12 h (user); hard stop 6 min before the budget end (outputs take seconds after it)
    "RERUN_ALL": {"source": "test", "n_tasks": None, "time_budget_sec": 12.0 * 3600 - 3 * 60,
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                  # v1.4.4: admit until the end (NVARC style); drain 8 min before the hard stop, admit 2 min before drain
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
}
if PROFILE not in PROFILE_TABLE:
    raise ValueError(f"unknown profile {PROFILE}; choose {list(PROFILE_TABLE)}")
P = PROFILE_TABLE[PROFILE]
SEED = 20261005
TASK_SAMPLE_SEED = 20261005

MODEL_PATH = Path(os.environ.get("Q27X_MODEL", "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"))
WHEELHOUSE = Path(os.environ.get("Q27X_WHEELHOUSE", "/kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3"))

# ---- generation budget (Kaggle discussion 745787: Qwen3.8 needs xhigh; median ~64k tokens; forced commit ~49k)
REASONING_EFFORT = "xhigh"
THINK_BUDGET = int(os.environ.get("Q27X_THINK_BUDGET", 60000))   # E11 rev 3: 60,000 = S1-60K (LB 34.58); user floor 60k   # S49: LB test of 49,152 (S1: 60,000; R5/E9: 77,824; floor 49,152)      # v1.2: NguyenThanhNhan forced the commit after ~49k thinking tokens
E11_THINK_FLOOR = 60000      # E11 rev 3 (user 2026-10-18): never below 60k (budget and admission guard)
assert THINK_BUDGET >= E11_THINK_FLOOR, (THINK_BUDGET, E11_THINK_FLOOR)
ANSWER_MAX_TOKENS = int(os.environ.get("Q27X_ANSWER_MAX", 8192))   # v1.4: grid-only answer; 49,152 + 8,192 = 57,344 = russcore's max_tokens
DRAIN_ANSWER_MAX_TOKENS = 8192                                      # v1.1 (F11): a 30x30 grid alone is ~2.7k tokens
STOP_THINKING_TEXT = "\n\nConsidering the limited time by the user, I have to give the solution based on the thinking directly now.\n</think>\n\n"
MAX_MODEL_LEN = int(os.environ.get("Q27X_MAX_MODEL_LEN", 106496))   # E9/E10: prompt (<= ~20k) + 77,824 + 8,192 fits; was 81,920   # v1.2: native context is 262k; fp8 KV makes room
CONTEXT_SAFETY = 256
MIN_THINK_BUDGET = 4096                  # v1.2: a long prompt gets a smaller thinking budget instead of being skipped
PROMPT_TOKEN_LIMIT = MAX_MODEL_LEN - ANSWER_MAX_TOKENS - CONTEXT_SAFETY - MIN_THINK_BUDGET
def phase1_cap(prompt_len):
    return max(0, min(THINK_BUDGET, MAX_MODEL_LEN - prompt_len - ANSWER_MAX_TOKENS - CONTEXT_SAFETY))
assert phase1_cap(MAX_MODEL_LEN) == 0 and phase1_cap(100) == min(THINK_BUDGET, MAX_MODEL_LEN - 100 - ANSWER_MAX_TOKENS - CONTEXT_SAFETY)
if (MAX_MODEL_LEN, THINK_BUDGET, ANSWER_MAX_TOKENS) == (106496, 77824, 8192):   # E9/E10
    assert phase1_cap(12800) == 77824 and phase1_cap(20000) == 77824 and phase1_cap(30000) == 68048
if (MAX_MODEL_LEN, THINK_BUDGET, ANSWER_MAX_TOKENS) == (81920, 49152, 8192):
    assert phase1_cap(24320) == 49152 and phase1_cap(30000) == 43472  # every real prompt (≤ ~9k tokens) keeps the full budget
SAMPLING = {"temperature": float(os.environ.get("Q27X_TEMPERATURE", 0.6)), "top_p": 0.95, "top_k": 20}   # v1.4: russcore LB-10.14 sampling

# ---- engine (PIV/V3.1 C1 settings; context, concurrency and MTP changed)
MAX_NUM_SEQS = int(os.environ.get("Q27X_MAX_NUM_SEQS", 16))   # upper bound; actual slots = KV tokens // EXPECTED_LIVE_TOKENS_PER_SEQ
MAX_NUM_BATCHED_TOKENS = 4096   # v1.3.5: = discussion max-prefill-tokens 4096; 32768 ran out of memory in run 2 (MTP drafter)
GPU_MEMORY_UTILIZATION = float(os.environ.get("Q27X_GPU_MEM", 0.90))   # v1.4.1: kept at 0.90 so prefix caching is the only change
ENABLE_PREFIX_CACHING = os.environ.get("Q27X_PREFIX_CACHE", "1") == "1"   # v1.4.1: first engine attempt (mamba 'align')
SPECULATIVE_CONFIG = {"method": "mtp", "num_speculative_tokens": int(os.environ.get("Q27X_SPEC_TOKENS", 3))}   # v1.2: vLLM Qwen3.8 recipe uses 3
KV_CACHE_DTYPE = os.environ.get("Q27X_KV_DTYPE", "fp8_e4m3")   # v1.3.4: back to the discussion recipe; run 1's crash was
# FlashInfer's JIT link step (`ld: cannot find -lcuda`), fixed in cell 5 (libcuda shim + link probe); fp8 KV itself loaded
# fine (312,000 KV tokens). Set Q27X_KV_DTYPE=auto for bf16 KV.
if os.environ.get("Q27X_NO_SPEC") == "1":
    SPECULATIVE_CONFIG = None
EXPECTED_LIVE_TOKENS_PER_SEQ = 43000      # a running task holds on average about half of its final ~60k length
# v1.3.5 correction: cell 5 divides the whole KV pool (run 2: 781 blocks × 1,600 = 1,249,600 tokens) by this. The pool is
# shared by 1 attention + 3 GDN groups; vLLM's "312,000 tokens" log line is pool // 4. A 30k-token task needs 19 attention
# + 12 GDN-state blocks (3 groups × (1 + 3 MTP)), so 16 tasks need ~496 of 781 blocks -> 16 slots (preemptions reported)

# ---- verification (fresh process per candidate: train first, then test)
VERIFY_TIMEOUT_SEC = 20.0
VERIFY_WORKERS = 4

# ---- time
TOTAL_BUDGET_SEC = float(os.environ.get("Q27X_TIME_BUDGET", P["time_budget_sec"]))
# v1.3: deadlines derived in a guaranteed order (v1.2 had DRAIN after HARD_STOP for EVAL12, so the drain never ran)
VERIFY_DEADLINE = NOTEBOOK_T0 + TOTAL_BUDGET_SEC - P.get("verify_margin_sec", 8 * 60)   # reporting only since v1.4
HARD_STOP = NOTEBOOK_T0 + TOTAL_BUDGET_SEC - P.get("hard_margin_sec", 25 * 60)        # abort everything, write outputs
DRAIN_DEADLINE = min(NOTEBOOK_T0 + TOTAL_BUDGET_SEC * P["drain_frac"], HARD_STOP - 8 * 60)    # force-close thinking
ADMIT_DEADLINE = min(NOTEBOOK_T0 + TOTAL_BUDGET_SEC * P["admit_frac"], DRAIN_DEADLINE - 7 * 60)  # no new task after this
if "drain_margin_sec" in P:   # v1.4.4 (RERUN): fixed offsets from the hard stop, so the slots stay full until the drain
    DRAIN_DEADLINE = HARD_STOP - P["drain_margin_sec"]
    ADMIT_DEADLINE = DRAIN_DEADLINE - P["admit_margin_sec"]
assert NOTEBOOK_T0 < ADMIT_DEADLINE < DRAIN_DEADLINE < HARD_STOP < VERIFY_DEADLINE < NOTEBOOK_T0 + TOTAL_BUDGET_SEC

# v1.3.5 (run 2 OOM): CUDA-graph memory is taken out of the KV budget (vLLM 0.19 envs.py 1654, v1/worker/gpu_worker.py
# 393-420); run 2 captured 0.61 GiB of graphs beyond it. Set here, before cell 5 spawns the vLLM workers (they inherit it).
os.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = "1"
# ---- E11-05: engine = SGLang 0.5.21 + RedHat INT4 (W4A16 g128) + DFlash2 drafter = the E11-04 W1 launch (2026-10-18).
# MODEL_PATH (FP8) stays: its tokenizer + chat template render every prompt exactly as in S49, and it is the vLLM fallback.
ENGINE_BACKEND = os.environ.get("E11_ENGINE", "sglang").strip().lower()       # sglang (vLLM FP8 fallback) | vllm (= S49)
E11_ALLOW_VLLM_FALLBACK = os.environ.get("E11_VLLM_FALLBACK", "1") == "1"
SGL_MODEL_PATH = Path(os.environ.get("E11_INT4", "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-int4-redhat/pytorch/w4a16-g128/1"))
SGL_DRAFT_PATH = Path(os.environ.get("E11_DFLASH", "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-dflash2/pytorch/bf16/1"))
SGL_WHEELHOUSE = Path(os.environ.get("E11_SGL_WHEELHOUSE", "/kaggle/input/datasets/foysalemonshanto/sglang-0-5-21-cu13-wheelhouse"))


def _e11_discover(path, marker, pred):
    """The given path if it holds `marker`, else the first /kaggle/input folder whose `marker` json satisfies pred."""
    if (path / marker).is_file():
        return path
    root = Path("/kaggle/input")
    for p in (sorted(root.rglob(marker)) if root.exists() else []):
        try:
            if pred(p.parent, _load_json(p)):
                return p.parent
        except Exception:
            continue
    return path


SGL_MODEL_PATH = _e11_discover(SGL_MODEL_PATH, "config.json", lambda d, c: isinstance(c, dict)
                               and (c.get("quantization_config") or {}).get("quant_method") == "compressed-tensors")
SGL_DRAFT_PATH = _e11_discover(SGL_DRAFT_PATH, "config.json", lambda d, c: isinstance(c, dict)
                               and c.get("architectures") == ["DFlash2DraftModel"])
SGL_WHEELHOUSE = _e11_discover(SGL_WHEELHOUSE, "BUILD_INFO.json", lambda d, c: isinstance(c, dict)
                               and c.get("sglang") == "0.5.21" and (d / "requirements.lock").is_file())
SGL_CONTEXT_LEN = 155648            # = the E11-04 W1 launch; every request stays inside MAX_MODEL_LEN (phase1_cap)
SGL_MAX_RUNNING = 16                # E15B (Notes/02 Rev 2c): 16 real streams; was 12 (E11-04 W1: 12 really running (mamba cache 56 / 4 state slots = 14 >= 12)
SGL_SERVER_MAX_RUNNING = 17         # E15B: client 16 + 1 (Notes/30 nodelay crash: a 13th on buffers sized 12); rev 3c: server admission limit = client 12 + 1. SGLang 0.5.21 counts the row of a request in
                                    # chunked prefill twice (scheduler.py:3971-3976) and latches batch_is_full (3846-3848) until a
                                    # running request ends -> EVAL30 ran 11 + 1 queued (2,358 of 2,509 decode lines). Upstream fix
                                    # PR #44401 (after 0.5.21); +1 is the documented workaround (PR #30295). CUDA graphs stay at 12.
SGL_LAUNCH_TIMEOUT_SEC = 30 * 60    # first launch includes JIT + CUDA graphs: 567 s in E11-04
SGL_RELAUNCH_TIMEOUT_SEC = 20 * 60  # a diagnosed relaunch (E11-04: about 75 s once the JIT cache exists)
SGL_GIVE_UP_AT = min(NOTEBOOK_T0 + (240 if RERUN else 75) * 60, ADMIT_DEADLINE - 5 * 60)   # E15B B5: RERUN 4 h   # no SGLang launch after this -> vLLM fallback
SGL_STREAM_IDLE_SEC = 1800          # a socket read silent for longer closes that request as 'error'
# rev 2 (GPT audit 2026-10-18, verified): capacity gate from the server log, fail-closed on missing evidence
SGL_MIN_RUNNING = 8                 # below this the SGLang engine is refused (-> S49 vLLM)
SGL_MIN_KV = 600_000                # KV tokens; E11-04 W1 had 881,336
SGL_KV_PER_REQUEST = 56_000         # E15P rev 2 (F6, agent B H1): 852,152 // 56k = 15 Poetiq streams. Poetiq units reach the 60k cap as a cohort: the E11-06 decode-line peak 0.73 of 866,744 = 51.9k per request x 16 = 0.97 of the E15B pool; simulated SGLang retractions at 16: 2 / 8 six-hour and 2 / 4 twelve-hour runs, at 15: 0 / 28 (-1 % tokens). The client KV gate cannot stop running units from growing. A smaller pool adapts (784k -> 14, 700k -> 12). Server mrr / graphs stay 17. E15B was 50k: the MEAN attempt context ~ 50k (attempt 1 at 45k ends at ~57-59k p90, max ~64k; attempt 2 less; Auditor 4); was 70k (KV // 70k = 12). concurrency <= KV // this (881,336 -> 12); a long Poetiq unit is prompt + 49k + 8k
SGL_MIN_FREE_GB = 1.0               # GPU headroom after start-up (E11-04 W1: 2.22 GB)
# rev 2: drain earlier. Aborted long prefixes were not cached in E11-04 (30k prefix: 0 cached, 24.5 s), so 12 phase-2
# continuations re-prefill up to ~0.7M tokens; 12 min instead of 8 min before the hard stop (SGLang only)
SGL_DRAIN_MARGIN_SEC = 12 * 60     # E15P: = S60 (S1-60K); E11-06 used 8 min
# rev 2: KV gate active on SGLang (S49's floor 12 = CONCURRENCY 12 made it inert): from 10 running, admit only below 85 %
SGL_KV_GATE_MIN_ACTIVE = 10
SGL_KV_ADMIT_MAX = 0.85

CONFIG = {k: globals()[k] for k in ["PROFILE", "RERUN", "SEED", "REASONING_EFFORT", "THINK_BUDGET", "ANSWER_MAX_TOKENS",
          "DRAIN_ANSWER_MAX_TOKENS", "MAX_MODEL_LEN", "PROMPT_TOKEN_LIMIT", "SAMPLING", "MAX_NUM_SEQS",
          "MAX_NUM_BATCHED_TOKENS", "GPU_MEMORY_UTILIZATION", "ENABLE_PREFIX_CACHING", "SPECULATIVE_CONFIG", "KV_CACHE_DTYPE", "VERIFY_TIMEOUT_SEC",
          "TOTAL_BUDGET_SEC", "MIN_THINK_BUDGET"]}
CONFIG["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = os.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"]
CONFIG["profile_row"] = P
CONFIG["e11"] = {k: (str(globals()[k]) if isinstance(globals()[k], Path) else globals()[k]) for k in (
    "ENGINE_BACKEND", "E11_ALLOW_VLLM_FALLBACK", "SGL_MODEL_PATH", "SGL_DRAFT_PATH", "SGL_WHEELHOUSE", "SGL_CONTEXT_LEN",
    "SGL_MAX_RUNNING", "SGL_SERVER_MAX_RUNNING", "SGL_LAUNCH_TIMEOUT_SEC", "SGL_RELAUNCH_TIMEOUT_SEC", "SGL_STREAM_IDLE_SEC", "SGL_MIN_RUNNING",
    "SGL_MIN_KV", "SGL_KV_PER_REQUEST", "SGL_MIN_FREE_GB", "SGL_DRAIN_MARGIN_SEC", "SGL_KV_GATE_MIN_ACTIVE", "SGL_KV_ADMIT_MAX")}
CONFIG["e11"]["give_up_after_sec"] = round(SGL_GIVE_UP_AT - NOTEBOOK_T0)
print("E11 engine:", json.dumps(CONFIG["e11"]))
print(json.dumps(CONFIG, indent=1, default=str))

os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
os.environ["HF_HUB_OFFLINE"] = "1"; os.environ["TRANSFORMERS_OFFLINE"] = "1"


# ---- E10 Poetiq donor configurations; source: poetiq-ai/poetiq-arc-agi-solver/config.py
# Original 86K context, GPU recipe, deadlines and original puzzle order above are unchanged.
POETIQ_NUM_EXPERTS = int(os.environ.get('E10_POETIQ_EXPERTS','1'))
POETIQ_MAX_ITERATIONS = int(os.environ.get('E10_POETIQ_MAX_ITERATIONS','10'))
POETIQ_MAX_SOLUTIONS = int(os.environ.get('E10_POETIQ_MAX_SOLUTIONS','5'))   # E15P: = S60 (S1-60K)
POETIQ_SOLVER_TEMPERATURE = float(os.environ.get('E10_POETIQ_TEMPERATURE','1.0'))
POETIQ_SOLVER_VARIANTS = tuple(int(v) for v in os.environ.get('E10_POETIQ_PROMPT_VARIANTS','1').split(','))
POETIQ_SELECTION_PROB = float(os.environ.get('E10_POETIQ_SELECTION_PROBABILITY','1.0'))
POETIQ_SEED = int(os.environ.get('E10_POETIQ_SEED','0'))
POETIQ_SHUFFLE_EXAMPLES = os.environ.get('E10_POETIQ_SHUFFLE','1')=='1'   # E15P: = S60 (S1-60K)
POETIQ_IMPROVING_ORDER = os.environ.get('E10_POETIQ_IMPROVING_ORDER','1')=='1'
POETIQ_USE_NEW_VOTING = os.environ.get('E10_POETIQ_NEW_VOTING','1')=='1'
POETIQ_COUNT_FAILED_MATCHES = os.environ.get('E10_POETIQ_COUNT_FAILED_MATCHES','1')=='1'
POETIQ_ITERS_TIEBREAK = os.environ.get('E10_POETIQ_ITERS_TIEBREAK','0')=='1'
POETIQ_LOW_TO_HIGH_ITERS = os.environ.get('E10_POETIQ_LOW_TO_HIGH_ITERS','0')=='1'
POETIQ_SANDBOX_TIMEOUT = float(os.environ.get('E10_POETIQ_SANDBOX_TIMEOUT','5'))
POETIQ_SANDBOX_WORKERS = int(os.environ.get('E10_POETIQ_SANDBOX_WORKERS','4'))
# S1: KV admission gate (no new request above this vLLM KV usage unless fewer than MIN_ACTIVE are running)
E10_KV_ADMIT_MAX = float(os.environ.get('E10_KV_ADMIT_MAX','0.95'))
E10_KV_GATE_MIN_ACTIVE = int(os.environ.get('E10_KV_GATE_MIN_ACTIVE','12'))
assert POETIQ_NUM_EXPERTS >= 1 and POETIQ_MAX_ITERATIONS >= 1
assert POETIQ_SOLVER_VARIANTS and all(v in (1,2,3) for v in POETIQ_SOLVER_VARIANTS)
assert POETIQ_MAX_SOLUTIONS >= 1 and 0 <= POETIQ_SELECTION_PROB <= 1
CONFIG['poetiq']={k:globals()[k] for k in (
    'POETIQ_NUM_EXPERTS','POETIQ_MAX_ITERATIONS','POETIQ_MAX_SOLUTIONS',
    'POETIQ_SOLVER_TEMPERATURE','POETIQ_SOLVER_VARIANTS','POETIQ_SELECTION_PROB',
    'POETIQ_SANDBOX_TIMEOUT','POETIQ_SANDBOX_WORKERS','POETIQ_USE_NEW_VOTING')}



# ---- E15P (user 2026-10-25): the Poetiq solver of E11-05 S60 (S1-60K, LB 34.58) on the E15B engine. No Duck: the Duck
# package, solver and report cells are not in this notebook. The engine cell reads one Duck name (the mamba-track test
# flag); it stays off as in every E15B run.
E12_MAMBA_TRACK_64 = False
# rev 2 (Notes/19): F2 mid-run server restarts (12-stream config, only with >= 45 min before the admission deadline),
# F5 sandbox self-test attempts (it runs while the model copy reads with 24 threads)
E15P_MAX_RESTARTS = int(os.environ.get("E15P_MAX_RESTARTS", 2))
E15P_RESTART_MIN_LEFT_SEC = float(os.environ.get("E15P_RESTART_MIN_LEFT_SEC", 45 * 60))
E15P_SELFTEST_TRIES = int(os.environ.get("E15P_SELFTEST_TRIES", 3))
CONFIG["e15p"] = {"solver": "poetiq-s60", "streams": SGL_MAX_RUNNING, "server_max_running": SGL_SERVER_MAX_RUNNING,
                  "kv_per_request": SGL_KV_PER_REQUEST, "drain_margin_sec": SGL_DRAIN_MARGIN_SEC, "rev": 2,
                  "max_restarts": E15P_MAX_RESTARTS, "restart_min_left_sec": E15P_RESTART_MIN_LEFT_SEC,
                  "engine_source": "6baa9361", "poetiq_source": "3af4660a"}
print("E15P:", json.dumps(CONFIG["e15p"]), "| poetiq", json.dumps(CONFIG["poetiq"]))


# ---- E17: engine-only; the rerun of this notebook only writes the cell-0 fallback
CONFIG["e17"] = {"on": bool(P.get("e17")) and not RERUN}
print("E17:", json.dumps(CONFIG["e17"]))


In [ ]:
# ==================================================================== CELL 1b — E14: first-line facts, phase record, staging
# Experiment 14 (Notes/07, 11-13). Runs before any heavy work: what the machine is (mounts / GCSFUSE, df, RAM, cgroup,
# CPU, a timed 64 MiB read), a phase record written at every change (e14_state.json) and a heartbeat, then the model
# copy to RAM/local disk with ranged parallel readers (verified, published atomically, fail open). The E14-P fix (raw
# reasoning in Duck's history, Notes/12) is switched on for every profile of this notebook.
import io as _e14_io
E14_DIR = Path(os.environ.get("E14_DIR", "/tmp/e14")); E14_DIR.mkdir(parents=True, exist_ok=True)
Q27X_PAYLOAD_SLUG = "esfersami50/q27-e15b-payload"


def q27x_payload(name, sha):
    """One file of the payload dataset (Notes/21: Kaggle commit runs of a 745 kB notebook never started, so the big
    payloads live in a dataset), found by its exact name under Q27X_PAYLOAD_DIR or /kaggle/input (at most 3 directory
    levels deep) and verified by sha256 before use."""
    roots = [Path(x) for x in os.environ.get("Q27X_PAYLOAD_DIR", "").split(os.pathsep) if x] + [Path("/kaggle/input")]
    seen = []
    for r in roots:
        for pat in (name, "*/" + name, "*/*/" + name, "*/*/*/" + name):
            for f in (sorted(r.glob(pat)) if r.is_dir() else []):
                b = f.read_bytes()
                if hashlib.sha256(b).hexdigest() == sha:
                    return b
                seen.append(str(f))
    raise FileNotFoundError(f"payload {name} (sha256 {sha[:12]}) not found or wrong under {[str(r) for r in roots]}; "
                            f"seen {seen}; attach the dataset {Q27X_PAYLOAD_SLUG}")


_E14_MOD_SHA = {"e14_loadpath.py": "7c66c179e5bb6bff7c909d0a96d4b7058c4bc0b2ff368b04367ee9f54e8fc5fc", "e14_measure.py": "ae61c9f9a03419d73dae2ba2d2aa4d5d1f4403780911832148d2dd8e8a3440d0", "e14_sglang_patches.py": "beb850abe2c072c34df0ae733642557cc44e2d808775912f291f072d29e12f11", "e14_nccl_bench.py": "355adda0d469c5ecac5abb94ff7608e2f256acc6a5dcf745e28b4d3676f03cb2", "e14_trace_summary.py": "8d89ca106e166c97204f1aeff931a50a75e203999c876f248b34e6ee24212b46"}
with zipfile.ZipFile(_e14_io.BytesIO(q27x_payload("q27x_e14_modules_55c1eb7911c6.bin", "55c1eb7911c68ba43d3c56b9c32bb00e581ccb96fe3b5b5548c5cdeb47125040"))) as _z:
    for _n, _h in _E14_MOD_SHA.items():
        (E14_DIR / _n).write_bytes(_z.read(_n))
        assert hashlib.sha256((E14_DIR / _n).read_bytes()).hexdigest() == _h, _n
sys.path.insert(0, str(E14_DIR))
import e14_loadpath as E14L, e14_measure as E14M, e14_sglang_patches as E14P, e14_trace_summary as E14T
E14_OUT = OUT / "e14"; E14_OUT.mkdir(parents=True, exist_ok=True)
E14_RESULTS = {"build": {"variant": "E17", "modules": _E14_MOD_SHA,
                         "payload": {"dataset": Q27X_PAYLOAD_SLUG, "q27x_e14_modules_55c1eb7911c6.bin": "55c1eb7911c68ba43d3c56b9c32bb00e581ccb96fe3b5b5548c5cdeb47125040"}},
               "phases": {}, "decisions": {}}
E14_DO_MEASURE = False   # E15P rev 2 (F10): no E14 measurement cells in this notebook
E14_STATE = E14L.E14State(WORKING / "e14_state.json")
E14_STATE.enter("PREPARE", deadline=HARD_STOP, profile=PROFILE, rerun=RERUN, measure=E14_DO_MEASURE)


def e14_save():
    try:
        write_json_atomic(E14_OUT / "e14_results.json", E14_RESULTS)
        write_json_atomic(WORKING / "e14_results.json", E14_RESULTS)
    except Exception as _e:
        print("E14 results write failed:", repr(_e)[:200], flush=True)


def _e14_heartbeat(every=60):
    while True:
        try:
            with open(WORKING / "e14_heartbeat.log", "a", encoding="utf-8") as _f:
                _f.write(f"{time.strftime('%H:%M:%S')} +{(time.perf_counter() - NOTEBOOK_T0) / 60:.1f} min "
                         f"phase {E14_STATE.rec.get('current')}\n")
        except Exception:
            pass
        time.sleep(every)


threading.Thread(target=_e14_heartbeat, daemon=True).start()
E14_RESULTS["env"] = E14L.env_report([str(SGL_MODEL_PATH), str(SGL_DRAFT_PATH), str(SGL_WHEELHOUSE), str(MODEL_PATH),
                                      "/kaggle/input"])
try:
    E14_RESULTS["read_probe"] = E14L.read_probe(SGL_MODEL_PATH) if ENGINE_BACKEND == "sglang" else None
except Exception as _e:
    E14_RESULTS["read_probe"] = {"error": repr(_e)[:200]}
print("E14 env:", json.dumps(E14_RESULTS["env"])[:1500], "| probe:", json.dumps(E14_RESULTS["read_probe"]), flush=True)
E14_STAGING = None
if ENGINE_BACKEND == "sglang" and E14L.loadpath_enabled():
    _e14_dest = tuple(x for x in os.environ.get("E14_DEST", "/dev/shm,/tmp").split(",") if x)
    E14_STAGING = E14L.Staging([SGL_MODEL_PATH, SGL_DRAFT_PATH], workers=24, stall_sec=600.0,
                               deadline=max(NOTEBOOK_T0 + 10 * 60, min(NOTEBOOK_T0 + 45 * 60, SGL_GIVE_UP_AT - 20 * 60)), log=print, state=E14_STATE,
                               dest_candidates=_e14_dest)   # E15B #27
    E14_STAGING.start()
    print("E14 staging started ->", _e14_dest, flush=True)
E14_VENV = {}


def e14_after_install(vpy, env):
    """After the SGLang venv exists: the NCCL bench (commit measurement only; the GPUs are idle while the copy runs),
    then the staged model paths (fail open: the original /kaggle/input paths)."""
    global SGL_MODEL_PATH, SGL_DRAFT_PATH
    E14_VENV.update(vpy=str(vpy), env=dict(env))
    if E14_DO_MEASURE and not E11_HARNESS_E14:
        E14_STATE.enter("NCCL_BENCH")
        _t = time.perf_counter()
        _o = E14_OUT / "nccl_bench.json"
        try:
            _r = subprocess.run([str(vpy), str(E14_DIR / "e14_nccl_bench.py"), "--out", str(_o),
                                 "--arms", "baseline,p2p_sys,shm_memcpy", "--timeout-per-arm", "240"],
                                env=env, capture_output=True, text=True, timeout=900)
            E14_RESULTS["nccl"] = json.loads(_o.read_text()) if _o.exists() else {"rc": _r.returncode,
                                                                                  "stderr": (_r.stderr or "")[-2000:]}
        except Exception as _e:
            E14_RESULTS["nccl"] = {"error": repr(_e)[:300]}
        E14_RESULTS["nccl_sec"] = round(time.perf_counter() - _t, 1)
        e14_save()
        print("E14 NCCL bench:", json.dumps((E14_RESULTS.get("nccl") or {}).get("summary"))[:800], flush=True)
    elif E14_DO_MEASURE and E11_HARNESS_E14:
        _o = E14_OUT / "nccl_bench.json"
        subprocess.run([sys.executable, str(E14_DIR / "e14_nccl_bench.py"), "--out", str(_o), "--fake"], timeout=120)
        E14_RESULTS["nccl"] = json.loads(_o.read_text()) if _o.exists() else {"error": "fake bench wrote nothing"}
    if E14_STAGING is not None:
        E14_STATE.enter("STAGING_JOIN")
        E14_STAGING.join(timeout=max(0.0, min(SGL_GIVE_UP_AT - 20 * 60, NOTEBOOK_T0 + 55 * 60) - time.perf_counter()))
        res = E14_STAGING.result
        if res is None:
            E14_RESULTS["staging"] = {"still_running": True, "note": "launch from the original paths"}
        else:
            paths, rep = res
            E14_RESULTS["staging"] = rep
            SGL_MODEL_PATH = Path(paths.get(str(SGL_MODEL_PATH), str(SGL_MODEL_PATH)))
            SGL_DRAFT_PATH = Path(paths.get(str(SGL_DRAFT_PATH), str(SGL_DRAFT_PATH)))
        E14_RESULTS["launch_paths"] = {"model": str(SGL_MODEL_PATH), "draft": str(SGL_DRAFT_PATH)}
        e14_save()
        print("E14 staging:", json.dumps(E14_RESULTS["staging"])[:1200], flush=True)
    E14_STATE.enter("LAUNCH", deadline=SGL_GIVE_UP_AT)


E11_HARNESS_E14 = os.environ.get("E11_HARNESS") == "1"
E14_TS = float(os.environ.get("E14_TIME_SCALE", "1"))    # the CPU harness shortens every window and phase cap; 1 on Kaggle
_ts = lambda s: max(1.5, float(s) * E14_TS)
CONFIG["e14"] = {"measure": E14_DO_MEASURE, "staging": E14_STAGING is not None,
                 "submit_arm": None if E14_DO_MEASURE else "DFlash width 6, 16 streams: mrr 17, graphs 1..17, mamba 68, min-free-slots-delay 1 (Exp 15 Notes/02)",
                 "modules": _E14_MOD_SHA}
e14_save()


In [ ]:
# ==================================================================== CELL 2 — offline install (PIV/V3.1 parity block) + tokenizer
import importlib, importlib.metadata
def pkg_version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None

EXPECTED_RUNTIME_PACKAGES = {"vllm": "0.19.0", "transformers": "4.57.6", "tokenizers": "0.22.2", "huggingface-hub": "0.36.2"}
_loaded = {"vllm": "vllm", "transformers": "transformers", "tokenizers": "tokenizers", "huggingface-hub": "huggingface_hub"}
before_stack = {n: pkg_version(n) for n in EXPECTED_RUNTIME_PACKAGES}

def offline_pip_install(requirements):
    if not requirements: return
    if not WHEELHOUSE.exists():
        raise FileNotFoundError(f"wheelhouse not attached: {WHEELHOUSE}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--no-cache-dir",
                           "--find-links", str(WHEELHOUSE), *requirements])

# S1: read the model files into the page cache in the background while pip runs (read-only, best effort)
if os.environ.get("Q27X_SKIP_INSTALL") != "1" and E14_STAGING is None:   # E14: ranged staging replaces the cat
    try:
        import shlex as _shlex
        _wdirs = (SGL_MODEL_PATH, SGL_DRAFT_PATH) if ENGINE_BACKEND == "sglang" else (MODEL_PATH,)   # E11-05
        _wfiles = sorted(str(p) for _d in _wdirs for p in Path(_d).glob("*.safetensors"))
        if _wfiles:
            subprocess.Popen(["bash", "-c", "cat " + " ".join(_shlex.quote(p) for p in _wfiles) + " > /dev/null"],
                             stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print("weight prefetch started:", len(_wfiles), "files")
    except Exception as _e:
        print("weight prefetch not started:", repr(_e)[:200])

if os.environ.get("Q27X_SKIP_INSTALL") != "1":     # Q27X_SKIP_INSTALL is for the local harness only
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError(f"Python {sys.version.split()[0]}: the cp312 wheelhouse needs the pinned Python 3.12 Kaggle image")
    for _pkg, _mod in _loaded.items():
        if before_stack.get(_pkg) != EXPECTED_RUNTIME_PACKAGES[_pkg] and _mod in sys.modules:
            raise RuntimeError(f"{_pkg} already imported at {before_stack.get(_pkg)!r}; restart the session before Run All.")
    _tok_pkgs = [f"{p}=={EXPECTED_RUNTIME_PACKAGES[p]}" for p in ("transformers", "tokenizers", "huggingface-hub")
                 if before_stack[p] != EXPECTED_RUNTIME_PACKAGES[p]]
    if ENGINE_BACKEND == "sglang" and before_stack["vllm"] != EXPECTED_RUNTIME_PACKAGES["vllm"]:
        # E11-05: the kernel only tokenizes (S49 stack); SGLang runs in its own venv; vLLM is installed only for the fallback
        try:
            offline_pip_install(_tok_pkgs)
        except Exception as _e:
            print("tokenizer-only install failed -> the full S49 install:", repr(_e)[:300], flush=True)
            offline_pip_install(["vllm==0.19.0"])
    elif before_stack["vllm"] != EXPECTED_RUNTIME_PACKAGES["vllm"]:
        offline_pip_install(["vllm==0.19.0"])
    else:
        offline_pip_install(_tok_pkgs)
    after_stack = {n: pkg_version(n) for n in EXPECTED_RUNTIME_PACKAGES}
    for _p, _want in EXPECTED_RUNTIME_PACKAGES.items():
        if _p == "vllm" and ENGINE_BACKEND == "sglang":
            continue                                   # E11-05: checked by the fallback cell, only if it runs
        if after_stack.get(_p) != _want:
            raise RuntimeError(f"runtime parity failure: {_p}={after_stack.get(_p)!r}, expected {_want!r}")
    print("runtime parity PASS:", after_stack)

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), trust_remote_code=False)
_model_cfg = _load_json(MODEL_PATH / "config.json")
assert _model_cfg.get("model_type") == "qwen3_5", _model_cfg.get("model_type")
assert (_model_cfg.get("quantization_config") or {}).get("quant_method") == "fp8"
THINK_CLOSE_IDS = tokenizer.encode("</think>", add_special_tokens=False)
assert len(THINK_CLOSE_IDS) == 1, THINK_CLOSE_IDS
THINK_CLOSE_ID = THINK_CLOSE_IDS[0]
STOP_IDS = tokenizer.encode(STOP_THINKING_TEXT, add_special_tokens=False)
assert THINK_CLOSE_ID in STOP_IDS
_gen_cfg = _load_json(MODEL_PATH / "generation_config.json") if (MODEL_PATH / "generation_config.json").exists() else {}
_eos = _gen_cfg.get("eos_token_id", [])
EOS_IDS = set(_eos if isinstance(_eos, list) else [_eos])     # v1.1 (F8): stripped before a forced close
print("tokenizer OK | </think> id", THINK_CLOSE_ID, "| stop text", len(STOP_IDS), "tokens")

# E11-05: the SGLang server receives token ids made by this (S49) tokenizer. Check that the INT4 checkpoint has the same
# vocabulary (with added tokens) and tokenizes a rendered prompt identically; otherwise run the S49 vLLM engine.
E11_TOKENIZER_CHECK = {"ok": None, "skipped": ENGINE_BACKEND != "sglang"}
if ENGINE_BACKEND == "sglang":
    try:
        from tokenizers import Tokenizer as _HFTokenizer
        _t4 = _HFTokenizer.from_file(str(SGL_MODEL_PATH / "tokenizer.json"))
        _probe = tokenizer.apply_chat_template(
            [{"role": "user", "content": "Example 1\nInput:\n0 1 2\n3 4 5\nOutput:\n5 4 3\n```python\ndef transform(grid):\n    return grid\n```"}],
            tokenize=False, add_generation_prompt=True, enable_thinking=True, reasoning_effort=REASONING_EFFORT)
        _probe += "Let me think.\n</think>\n\n```python\nimport numpy as np\n```<|im_end|>"
        # rev 2: + a real ARC prompt fragment (every train pair of the first test task, rendered as digit rows)
        _t0 = {"train": [{"input": [[0, 1, 2], [3, 4, 5]], "output": [[5, 4, 3], [2, 1, 0]]}]}   # E15B #17: fixed
        _probe += "".join(f"\nExample {_i + 1}\nInput:\n" + "\n".join(" ".join(str(v) for v in _row) for _row in _p["input"])
                          + "\nOutput:\n" + "\n".join(" ".join(str(v) for v in _row) for _row in _p["output"])
                          for _i, _p in enumerate(_t0["train"]))
        _ids_s49 = tokenizer.encode(_probe, add_special_tokens=False)
        _ids_int4 = _t4.encode(_probe, add_special_tokens=False).ids
        _same_vocab = _t4.get_vocab(with_added_tokens=True) == tokenizer.backend_tokenizer.get_vocab(with_added_tokens=True)
        # rev 2: the BPE merges too (the two files store them in different formats: "a b" strings vs [a, b] pairs)
        _norm = lambda m: [tuple(x.split(" ", 1)) if isinstance(x, str) else tuple(x) for x in m]
        _same_merges = (_norm(_load_json(SGL_MODEL_PATH / "tokenizer.json")["model"]["merges"])
                        == _norm(_load_json(MODEL_PATH / "tokenizer.json")["model"]["merges"]))
        _int4_gen = _load_json(SGL_MODEL_PATH / "generation_config.json") if (SGL_MODEL_PATH / "generation_config.json").exists() else {}
        _e4 = _int4_gen.get("eos_token_id", [])
        _same_eos = set(_e4 if isinstance(_e4, list) else [_e4]) == EOS_IDS
        E11_TOKENIZER_CHECK = {"ok": bool(_same_vocab and _same_merges and _ids_s49 == _ids_int4 and _same_eos),
                               "same_vocab": _same_vocab, "same_merges": _same_merges, "same_probe_ids": _ids_s49 == _ids_int4,
                               "probe_tokens": len(_ids_s49), "same_eos": _same_eos}
    except Exception as _e:
        E11_TOKENIZER_CHECK = {"ok": False, "error": f"{type(_e).__name__}: {_e}"[:300]}
    if not E11_TOKENIZER_CHECK["ok"]:
        print("E11 tokenizer check FAILED -> S49 vLLM FP8 engine:", E11_TOKENIZER_CHECK, flush=True)
        ENGINE_BACKEND = "vllm"
CONFIG["e11"]["tokenizer_check"] = E11_TOKENIZER_CHECK
print("E11 tokenizer check:", E11_TOKENIZER_CHECK)


In [ ]:
# E10: original task-order and evaluation profiles, but one Poetiq program per task/expert/iteration.
if P['source']=='eval':
    _ch=_load_json(ARC_ROOT/'arc-agi_evaluation_challenges.json')
    _sol=_load_json(ARC_ROOT/'arc-agi_evaluation_solutions.json')
    if P.get('task_list'):
        assert all(t in _ch for t in P['task_list']), 'Evaluation task ID missing'
        TASK_IDS=sorted(P['task_list'])
    else:
        TASK_IDS=sorted(random.Random(TASK_SAMPLE_SEED).sample(sorted(_ch),P['n_tasks']))
    CHALLENGES={t:_ch[t] for t in TASK_IDS}
    SOLUTIONS={t:_sol[t] for t in TASK_IDS}
else:
    CHALLENGES=dict(test_challenges)
    TASK_IDS=sorted(CHALLENGES) # no Codex ordering
    SOLUTIONS=None
N_HIDDEN_UNITS=sum(len(t['test']) for t in test_challenges.values())
print('E10 Poetiq:',len(TASK_IDS),'tasks,',sum(len(CHALLENGES[t]['test']) for t in TASK_IDS),
      'queries | experts',POETIQ_NUM_EXPERTS,'iterations per expert',POETIQ_MAX_ITERATIONS)


In [ ]:
# ==================================================================== E17: no solver in this notebook
# The engine cell starts SGLang only after the solver's sandbox self-test; E17 runs no generated code.
E10_SANDBOX_SELFTEST = {"ok": True, "mode": "E17 engine tests: no solver, no sandbox"}


In [ ]:
# E17: the production engine cell, unchanged (its launch = the first, JIT-warming launch). E15P: the E15B engine cell (Poetiq uses /generate through the shim; the chat parsers of the Duck flags do not touch
# /generate). rev 2 (Notes/19): F3 pending deltas for the drain, F4 admission usage = max(KV, mamba), F15 a smoke
# failure relaunches the 12-stream config once, F2 e15p_relaunch_s12() for a mid-run restart, F12 pool maxima.
# E12: = E11-06 CELL 5 unchanged except the Duck serving flags in e11_flags().
# ==================================================================== CELL 5 — engine: SGLang 0.5.21 + RedHat INT4 + DFlash2 (E11-04 W1)
# E11-05. The scheduler (next cells) is S49's, unchanged. It drives the engine with the vLLM calls it already uses
# (add_request / step / abort_request, DELTA token ids, finish 'stop' | 'length' | 'abort'); the shim below maps them onto
# SGLang's HTTP API: /generate streamed with --incremental-streaming-output (disjoint deltas, tokenizer_manager.py:2493-2515),
# /abort_request (the stream then ends with finish type 'abort'), /metrics. Launch = E11-04 W1, measured 2026-10-18.
# If SGLang does not come up (inputs, install, launch, smoke incl. abort), the server is killed and CELL 5b runs S49's vLLM.
import queue as _queue, signal as _signal, statistics as _statistics, types as _types, urllib.request as _urlreq
E11_HARNESS = os.environ.get("E11_HARNESS") == "1"            # local CPU harness only: fake server, no install, no GPU
SGL_PORT = int(os.environ.get("E11_PORT", 30000))
SGL_URL = f"http://127.0.0.1:{SGL_PORT}"
E11_SGL = {"requested": ENGINE_BACKEND, "ok": False, "stage": "not started", "inputs": {}, "install": None, "attempts": [],
           "caps": None, "launch_flags": None, "smoke": None, "error": None, "tokenizer_check": E11_TOKENIZER_CHECK}
E11_SERVER = None
ENGINE_READY = False; llm = None; engine = None; CONCURRENCY = 0
ENGINE_INFO = {"backend": None, "sglang": E11_SGL, "kv_tokens": None, "concurrency": None, "fatal": None, "load_sec": None}


class SamplingParams:
    """vLLM-shaped container. The shim sends max_tokens, temperature, top_p, top_k. The seed is not sent: SGLang 0.5.21
    uses sampling_seed only under --enable-deterministic-inference (sampling_batch_info.py:143-157), so sampling is random."""
    def __init__(self, max_tokens=16, seed=None, temperature=1.0, top_p=1.0, top_k=-1, **kw):
        self.max_tokens, self.seed = int(max_tokens), seed
        self.temperature, self.top_p, self.top_k = float(temperature), float(top_p), int(top_k)
        self.extra = kw


class RequestOutputKind:
    DELTA = "delta"


def make_params(max_tokens, seed):
    return SamplingParams(max_tokens=int(max_tokens), seed=int(seed), detokenize=False, skip_special_tokens=False,
                          output_kind=RequestOutputKind.DELTA, **SAMPLING)


class E11Fail(Exception):
    pass


def _e11_post(path, payload, timeout=30):
    req = _urlreq.Request(SGL_URL + path, data=json.dumps(payload).encode(), method="POST",
                          headers={"Content-Type": "application/json"})
    with _urlreq.urlopen(req, timeout=timeout) as r:
        return r.read()


def _e11_gpu_used():
    try:
        return [int(x) for x in subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                                               capture_output=True, text=True, timeout=30).stdout.split()]
    except Exception:
        return []


def _e11_gpu_free(max_sec=180):
    t = time.perf_counter()
    while time.perf_counter() - t < max_sec:
        u = _e11_gpu_used()
        if E11_HARNESS or (u and max(u) < 1024):
            return {"free": True, "sec": round(time.perf_counter() - t), "used_mb": u}
        time.sleep(5)
    for _pat in ("sglang.launch_server", "sglang::"):
        subprocess.run(["pkill", "-9", "-f", _pat], capture_output=True)
    time.sleep(15)
    return {"free": False, "sec": round(time.perf_counter() - t), "used_mb": _e11_gpu_used(), "pkill": True}


# ---- install (= E11-03/04 C1): a venv from the wheelhouse lock, --no-deps; CUDA 13 pip runtime as CUDA_HOME; link shims
def e11_install():
    if E11_HARNESS:
        return Path(sys.executable), dict(os.environ), {"harness": True}
    shims, t0 = {}, time.perf_counter()
    venv = Path("/tmp/sglang-venv"); vpy = venv / "bin" / "python"
    # Kaggle drops '+' from dataset file names (flashinfer_jit_cache-0.6.18+cu130 -> ...0.6.18cu130); rebuild the original
    # names from SHA256SUMS as links in a find-links folder, checking the hash of every renamed wheel
    fl = Path("/tmp/e11-wheels"); fl.mkdir(parents=True, exist_ok=True)
    renamed = []
    for line in (SGL_WHEELHOUSE / "SHA256SUMS").read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        h, name = line.split("  ", 1); name = name.strip()
        if not name.endswith(".whl"):
            continue
        s = SGL_WHEELHOUSE / name if (SGL_WHEELHOUSE / name).exists() else SGL_WHEELHOUSE / name.replace("+", "")
        if not s.exists():
            raise E11Fail(f"wheel missing from the dataset: {name}")
        if s.name != name:
            d = hashlib.sha256()
            with open(s, "rb") as f:
                for b in iter(lambda: f.read(16 * 1024 * 1024), b""):
                    d.update(b)
            if d.hexdigest() != h:
                raise E11Fail(f"sha256 mismatch for {s.name}")
            renamed.append(f"{s.name} -> {name}")
        if not (fl / name).exists():
            (fl / name).symlink_to(s)
    shims["renamed_wheels"] = renamed
    if not vpy.exists():
        subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(venv)], check=True, timeout=300)
        r = subprocess.run([sys.executable, "-m", "pip", "--python", str(vpy), "install", "--no-index", "--find-links", str(fl),
                            "--requirement", str(SGL_WHEELHOUSE / "requirements.lock"), "--no-deps", "--only-binary", ":all:",
                            "--no-compile", "--disable-pip-version-check", "--no-warn-conflicts", "-q"],
                           capture_output=True, text=True, timeout=1800)
        if r.returncode != 0:
            raise E11Fail("offline SGLang install failed: " + (r.stderr or r.stdout)[-1500:])
    site = Path(subprocess.run([str(vpy), "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"],
                               capture_output=True, text=True, check=True, timeout=120).stdout.strip())
    cuda_home = site / "nvidia" / "cu13"
    # E11-02 BUILD_PROBE: the pip CUDA runtime ships lib/libcudart.so.13 only; JIT links need libcudart.so and lib64
    if not (cuda_home / "lib" / "libcudart.so").exists():
        (cuda_home / "lib" / "libcudart.so").symlink_to("libcudart.so.13"); shims["libcudart.so"] = "created"
    if not (cuda_home / "lib64").exists():
        (cuda_home / "lib64").symlink_to("lib"); shims["lib64"] = "created"
    # E10 run 1: FlashInfer's JIT link needs -lcuda; Kaggle has only libcuda.so.1
    libcuda = Path("/tmp/e11-libcuda")
    have = [d for d in ("/usr/local/cuda/lib64", "/usr/local/cuda/lib64/stubs", "/usr/lib/x86_64-linux-gnu", "/usr/lib64")
            if Path(d, "libcuda.so").exists()]
    if have:
        shims["libcuda.so"] = f"exists in {have[0]}"
    else:
        real = None
        try:
            for line in subprocess.run(["ldconfig", "-p"], capture_output=True, text=True, timeout=30).stdout.splitlines():
                if "libcuda.so.1" in line and "=>" in line:
                    real = line.split("=>")[-1].strip(); break
        except Exception:
            pass
        for d in ("/usr/local/nvidia/lib64", "/usr/lib/x86_64-linux-gnu", "/usr/lib64"):
            if real is None and Path(d, "libcuda.so.1").exists():
                real = str(Path(d, "libcuda.so.1"))
        if real:
            libcuda.mkdir(parents=True, exist_ok=True)
            if (libcuda / "libcuda.so").is_symlink() or (libcuda / "libcuda.so").exists():
                (libcuda / "libcuda.so").unlink()
            (libcuda / "libcuda.so").symlink_to(Path(real).resolve()); shims["libcuda.so"] = f"shim -> {real}"
        else:
            shims["libcuda.so"] = "libcuda.so.1 NOT FOUND"
    env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "CUDA_HOME", "CUDA_PATH")}
    env.update({
        "PATH": os.pathsep.join([str(venv / "bin"), str(cuda_home / "bin"), os.environ.get("PATH", "")]),
        "CUDA_HOME": str(cuda_home), "CUDA_PATH": str(cuda_home),
        "LIBRARY_PATH": os.pathsep.join(x for x in (str(libcuda), str(cuda_home / "lib"), os.environ.get("LIBRARY_PATH", "")) if x),
        "LD_LIBRARY_PATH": os.pathsep.join(x for x in (str(cuda_home / "lib"), os.environ.get("LD_LIBRARY_PATH", "")) if x),
        "TORCH_CUDA_ARCH_LIST": "8.9", "MAX_JOBS": "4",
        "SGLANG_CACHE_DIR": "/tmp/e11-cache/sglang", "CUDA_CACHE_PATH": "/tmp/e11-cache/cuda",
        "TRITON_CACHE_DIR": "/tmp/e11-cache/triton", "TORCH_EXTENSIONS_DIR": "/tmp/e11-cache/torch_ext",
        "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "TOKENIZERS_PARALLELISM": "false",
        "CUDA_DEVICE_ORDER": "PCI_BUS_ID", "PYTHONUNBUFFERED": "1",
    })
    env.update(E14L.jit_cache_env())      # E14: SGLANG_JIT_CACHE_DIR set explicitly (it is not under SGLANG_CACHE_DIR)
    shims.update(cuda_home=str(cuda_home), venv_python=str(vpy), sec=round(time.perf_counter() - t0, 1))
    return vpy, env, shims


# ---- launch: the E11-04 W1 command; one diagnosed fallback per cause (E11-04 rev 2b next_launch)
W1 = {"strategy": "extra_buffer_lazy", "mamba_cache": 68, "no_prefill_graph": True, "mem": "0.83"}   # E15B: 16 x ~3 slots + radix
E11_OOM_MARKERS = ("out of memory", "outofmemory", "cuda error: out of memory", "not enough memory", "insufficient memory",
                   "cudaerrormemoryallocation")


def e11_flags(cfg):
    f = ["--model-path", str(SGL_MODEL_PATH), "--tp-size", "4", "--kv-cache-dtype", "fp8_e4m3",
         "--context-length", str(SGL_CONTEXT_LEN), "--max-running-requests", str(SGL_SERVER_MAX_RUNNING),
         "--chunked-prefill-size", "4096", "--mamba-ssm-dtype", "float16", "--enable-metrics", "--disable-flashinfer-autotune",
         "--random-seed", str(SEED), "--host", "127.0.0.1", "--port", str(SGL_PORT), "--log-level", "info",
         "--cuda-graph-max-bs-decode", str(SGL_MAX_RUNNING), "--watchdog-timeout", "1800", "--decode-log-interval", "40",
         "--speculative-algorithm", "DFLASH", "--speculative-draft-model-path", str(SGL_DRAFT_PATH),
         "--speculative-draft-model-quantization", "unquant", "--speculative-num-draft-tokens", "8"]
    if cfg["no_prefill_graph"]:
        f += ["--disable-prefill-cuda-graph"]
    f += ["--mamba-radix-cache-strategy", cfg["strategy"], "--max-mamba-cache-size", str(cfg["mamba_cache"]),
          "--mem-fraction-static", cfg["mem"],
          # E11-05: streaming for the shim (disjoint deltas). rev 2: interval 1 = a chunk every decode step. With N > 1 the
          # rule is len(output_ids) % N == 1 (output_streamer.py:427-436); DFLASH accepts several tokens per step, so a
          # stream could skip that residue for long stretches and the drain would lose the unsent thinking.
          "--incremental-streaming-output", "--stream-interval", "1"]
    # E12: Duck milestone-2 serving flags (dfranzen notebook cell 12; each verified in SGLang 0.5.21): OpenAI tool calls
    # and reasoning split for /v1/chat/completions, preserved thinking by default, prefix-aware scheduling, cached_tokens
    # in usage. /generate (the smoke below) is unaffected.
    f += ["--reasoning-parser", "qwen3", "--tool-call-parser", "qwen3_coder",
          "--default-chat-template-kwargs", json.dumps({"preserve_thinking": True}),
          "--schedule-policy", "lpm", "--enable-cache-report"]
    if E12_MAMBA_TRACK_64:
        f += ["--mamba-track-interval", "64", "--page-size", "64"]
    return f


# E14S (Notes/30, R2): the adopted arm = DFlash width 6 (531 vs 505 tok/s, gate PASS) on CUDA graphs 1..13 (captured
# 12 and 13, gate PASS). Set in every launch unless the E14A measurements run (they start from W1 and set each arm per
# launch).
E14_SUBMIT_ARM = {"--cuda-graph-bs-decode": [str(b) for b in range(1, SGL_SERVER_MAX_RUNNING + 1)],
                  "--speculative-num-draft-tokens": ["6"],
                  # E15B #36 (Exp 15 Notes/02): the DFlash min-free-slots delayer off ("<= 1 disables",
                  # managers/min_free_slots_delayer.py): with it the last stream waited while the others ran (EVAL120:
                  # 11.2 of 12 running, queue > 0 for 72 % of the time)
                  "--min-free-slots-delay": ["1"]}
E14_SUBMIT_DROP = ("--cuda-graph-max-bs-decode",)
_e11_flags_w1 = e11_flags


def e11_flags(cfg):
    f = _e11_flags_w1(cfg)
    if E14_DO_MEASURE or cfg.get("w1"):     # E15B #16: the W1 fallback launch keeps the plain W1 flags
        return f
    out, i, names = [], 0, set(E14_SUBMIT_ARM) | set(E14_SUBMIT_DROP)
    while i < len(f):
        if f[i] in names:
            i += 1
            while i < len(f) and not str(f[i]).startswith("--"):
                i += 1
            continue
        out.append(f[i])
        i += 1
    for k, v in E14_SUBMIT_ARM.items():
        out += [k] + list(v)
    if cfg.get("streams12"):     # E15B #16: the 12-stream fallback launch (graphs 1..13, mrr 13; the delayer stays off)
        out = e14_edit_flags_b(out, {"--max-running-requests": ["13"], "--cuda-graph-bs-decode": [str(b) for b in range(1, 14)]})
    assert "--enable-prefill-delayer" not in out
    return out


def e14_edit_flags_b(flags, set_):
    out, i = [], 0
    while i < len(flags):
        if flags[i] in set_:
            i += 1
            while i < len(flags) and not str(flags[i]).startswith("--"):
                i += 1
            continue
        out.append(flags[i])
        i += 1
    for k, v in set_.items():
        out += [k] + list(v)
    return out


def e11_next(cfg, tail, used):
    """(cfg, reason) for the one diagnosed relaunch of this cause, or None. Lazy / prefill causes are read from error
    lines only (normal startup lines also say 'Mamba Cache ...')."""
    t = "\n".join(l for l in (tail or "").splitlines()
                  if any(k in l.lower() for k in ("error", "exception", "assert", "unsupported", "not supported")))[-4000:].lower()
    global SGL_MAX_RUNNING, SGL_SERVER_MAX_RUNNING
    _oom = any(k in (tail or "").lower() for k in E11_OOM_MARKERS)
    if not cfg.get("streams12") and "streams12" not in used:   # E15B #16 (H2): ANY failed 16-stream launch
        # E15B #16: a 16-stream launch that ran out of memory or failed without a diagnosed cause relaunches at the
        # E14S 12-stream size (mamba 56, mrr 13, graphs 1..13; the delayer stays off); the capacity gate then sees 12
        SGL_MAX_RUNNING, SGL_SERVER_MAX_RUNNING = 12, 13
        return dict(cfg, streams12=True, mamba_cache=56), "streams12"
    if _oom and cfg["mem"] == "0.83" and "oom" not in used:
        return dict(cfg, mem="0.78"), "oom"
    if cfg["strategy"] == "extra_buffer_lazy" and ("lazy" in t or "extra_buffer" in t or "mamba" in t) and "lazy" not in used:
        return dict(cfg, strategy="extra_buffer", mamba_cache=5 * SGL_SERVER_MAX_RUNNING), "lazy"   # E15B: 5 slots x mrr
    if cfg["no_prefill_graph"] and "graph" in t and "prefill" not in used:
        return dict(cfg, no_prefill_graph=False), "prefill"
    if "w1" not in used:          # E15B #16 (H2): last resort, the plain W1 flags (width 8, the server's defaults)
        return dict(cfg, w1=True), "w1"
    return None


def e11_slots(cfg):
    """kv_cache_configurator.py:2232-2261 with the overlap scheduler on: extra_buffer 5, extra_buffer_lazy 4 slots."""
    return {"extra_buffer_lazy": 4, "no_buffer": 3}.get(cfg["strategy"], 5)


def e11_caps(text):
    """Effective limits from the server log (/get_server_info reports the requested values)."""
    caps = {}
    m = re.findall(r"max_running_requests is capped to (\d+) by the mamba state cache \(max_mamba_cache_size=(\d+), (\d+) state", text)
    caps["capped_to"] = min(int(a) for a, _, _ in m) if m else None
    tp0 = "\n".join(l for l in text.splitlines() if "TP0]" in l) or text
    kv = re.findall(r"KV Cache is allocated\. dtype: ([\w.]+), #tokens: (\d+), K size: ([\d.]+) GB, V size: ([\d.]+) GB", tp0)
    caps["kv_tokens"] = int(kv[0][1]) if kv else None                     # first TP0 line = target (second = drafter)
    mc = re.findall(r"Mamba Cache is allocated\. max_mamba_cache_size: (\d+)", tp0)
    caps["mamba_cache_size"] = int(mc[0]) if mc else None
    vb = re.findall(r"Capture target verify CUDA graph begin\..*?bs=\[([\d, ]+)\]", text)
    caps["verify_graph_bs"] = [int(x) for x in re.findall(r"\d+", vb[0])] if vb else None
    caps["prefill_graph_captured"] = "Capture target prefill CUDA graph" in text
    am = re.findall(r"avail mem=([\d.]+) GB", tp0)
    caps["avail_mem_gb_last"] = float(am[-1]) if am else None
    return caps


def e11_capacity_gate(caps, cfg):
    """rev 2 (GPT audit): fail-closed. Every limit must be in the server log; concurrency = the smallest of 12, the mamba
    cache // state slots (and the logged cap), the largest captured verify-graph batch <= 12, and KV // 70k."""
    slots = e11_slots(cfg)
    mc, kv, free = caps.get("mamba_cache_size"), caps.get("kv_tokens"), caps.get("avail_mem_gb_last")
    eff = min(SGL_MAX_RUNNING, mc // slots) if isinstance(mc, int) else None
    if eff is not None and caps.get("capped_to"):
        eff = min(eff, caps["capped_to"])
    bs = [b for b in (caps.get("verify_graph_bs") or []) if b <= SGL_MAX_RUNNING]
    caps.update(slots_per_request=slots, effective_running=eff)
    conc = min(SGL_MAX_RUNNING, eff or 0, max(bs) if bs else 0, (kv or 0) // SGL_KV_PER_REQUEST)
    g = {"mamba_cache_logged": eff is not None, "kv_logged": kv is not None, "verify_graph_logged": bool(bs),
         "running_min": (eff or 0) >= SGL_MIN_RUNNING, "kv_min": (kv or 0) >= SGL_MIN_KV,
         "prefill_graph_as_requested": (caps.get("prefill_graph_captured") is False) if cfg["no_prefill_graph"] else True,
         "headroom": isinstance(free, (int, float)) and free >= SGL_MIN_FREE_GB, "concurrency_min": conc >= SGL_MIN_RUNNING}
    g["pass"] = all(g.values())
    return g, conc


class E11Server:
    def __init__(self, cmd, env, tag):
        self.cmd, self.env, self.tag, self.proc, self.log_f = cmd, env, tag, None, None
        self.log_path = OUT / f"sglang_server_{tag}.log"

    def start(self):
        self.log_f = open(self.log_path, "w", encoding="utf-8")
        self.t0 = time.perf_counter()
        self.proc = subprocess.Popen(self.cmd, env=self.env, stdout=self.log_f, stderr=subprocess.STDOUT,
                                     cwd=None if E11_HARNESS else "/tmp", start_new_session=True)

    def alive(self):
        return self.proc is not None and self.proc.poll() is None

    def log_text(self):
        try:
            return self.log_path.read_text(encoding="utf-8", errors="replace")
        except Exception:
            return ""

    def wait_ready(self, timeout):
        # E14-0 (Notes/07, 11): stall-based. Progress = log growth or the server tree's read() bytes + page faults (the
        # mmap weight load is silent in the log); stalls: 900 s while loading, 720 s while compiling (> the ~6 min silent
        # first verify-graph capture); a progressing server is never given up before t0 + timeout
        def _health():
            _urlreq.urlopen(SGL_URL + "/health", timeout=10).read()
            return True
        mon = E14L.StallMonitor(self.proc, self.log_path, poll_sec=2 if E11_HARNESS else 5, log=print)
        ok, why, self.e14_stats = mon.wait(_health, give_up_at=self.t0 + timeout)
        return ok, why

    def kill(self, wait=True):
        if self.proc is not None and not self.alive() and hasattr(os, "killpg"):    # E15B B4: orphaned TP workers
            try:
                os.killpg(self.proc.pid, getattr(_signal, "SIGKILL", _signal.SIGTERM))
            except Exception:
                pass
        if self.alive():
            for sig, w in ((_signal.SIGTERM, 30 if wait else 0), (getattr(_signal, "SIGKILL", _signal.SIGTERM), 30)):
                try:
                    if hasattr(os, "killpg"):
                        os.killpg(self.proc.pid, sig)          # the whole group: TP workers, scheduler, detokenizer
                    else:
                        self.proc.kill()
                except Exception:
                    pass
                try:
                    self.proc.wait(timeout=w if w else 0.1)
                    break
                except Exception:
                    continue
        if self.log_f:
            try:
                self.log_f.close()
            except Exception:
                pass
            self.log_f = None


def e11_kill_server(wait=True):
    if E11_SERVER is not None:
        E11_SERVER.kill(wait=wait)


# ---- the shim: the vLLM engine calls of the S49 scheduler on top of SGLang's HTTP API
class _E11Completion:
    __slots__ = ("token_ids", "finish_reason")

    def __init__(self, ids, reason):
        self.token_ids, self.finish_reason = ids, reason


class _E11Output:
    __slots__ = ("request_id", "outputs", "finished")

    def __init__(self, rid, ids, reason):
        self.request_id, self.outputs, self.finished = rid, [_E11Completion(ids, reason)], reason is not None


class E11SglangEngine:
    """add_request starts one streaming /generate per request in a thread; step() returns the token deltas that arrived
    (merged per request, order kept) and raises when the server died; abort_request posts /abort_request per id."""

    def __init__(self, server):
        self.server, self.q, self.aborted, self.done = server, _queue.Queue(), set(), {}
        self.held = []                      # E15P rev 2 (F3): items of other requests set aside by take_pending()
        self.lock = threading.Lock()
        self.stats = {"added": 0, "finished": 0, "by_reason": {}, "errors": [], "prompt_tokens": 0, "cached_tokens": 0,
                      "completion_tokens": 0, "aborts_sent": 0, "abort_errors": 0, "abort_timeouts": 0,
                      "abort_wait_sec_max": 0.0, "accept_len_sum": 0.0, "accept_len_n": 0}
        self.req_log = OUT / "sglang_requests.jsonl"

    def add_request(self, rid, prompt, params):
        if not self.server.alive():
            raise RuntimeError(f"SGLang server is not running (rc={self.server.proc.returncode})")
        sp = {"max_new_tokens": int(params.max_tokens), "temperature": float(params.temperature), "top_p": float(params.top_p),
              "top_k": int(params.top_k), "skip_special_tokens": False, "no_stop_trim": True}
        body = json.dumps({"rid": rid, "input_ids": [int(x) for x in prompt["prompt_token_ids"]],
                           "sampling_params": sp, "stream": True}).encode()
        with self.lock:
            self.stats["added"] += 1
        threading.Thread(target=self._stream, args=(rid, body, time.perf_counter()), daemon=True).start()

    def _stream(self, rid, body, t0):
        reason, meta, err, n = None, {}, None, 0
        try:
            req = _urlreq.Request(SGL_URL + "/generate", data=body, method="POST", headers={"Content-Type": "application/json"})
            with _urlreq.urlopen(req, timeout=SGL_STREAM_IDLE_SEC) as r:
                for raw in r:
                    line = raw.strip()
                    if not line.startswith(b"data:"):
                        continue
                    data = line[5:].strip()
                    if data == b"[DONE]":
                        break
                    d = json.loads(data)
                    if "error" in d:
                        err = str(d["error"])[:300]
                        break
                    ids = d.get("output_ids") or []
                    m = d.get("meta_info") or {}
                    fr = m.get("finish_reason")
                    if fr is not None and reason is None:
                        reason = (fr.get("type") if isinstance(fr, dict) else str(fr)) or "stop"
                        meta = m
                    n += len(ids)
                    if ids or reason is not None:
                        self.q.put((rid, list(ids), reason))
                    if reason is not None:
                        break
        except Exception as e:
            err = f"{type(e).__name__}: {e}"[:300]
        if reason is None:
            reason = "error"
            self.q.put((rid, [], "error"))
        rec = {"rid": rid, "finish": reason, "tokens": n, "sec": round(time.perf_counter() - t0, 1),
               "prompt_tokens": meta.get("prompt_tokens"), "cached_tokens": meta.get("cached_tokens"),
               "completion_tokens": meta.get("completion_tokens"), "spec_accept_length": meta.get("spec_accept_length"),
               "error": err}
        with self.lock:
            self.done[rid] = reason                     # rev 2: the stream has ended (abort_request waits for this)
            s = self.stats
            s["finished"] += 1; s["by_reason"][reason] = s["by_reason"].get(reason, 0) + 1
            s["prompt_tokens"] += int(meta.get("prompt_tokens") or 0); s["cached_tokens"] += int(meta.get("cached_tokens") or 0)
            s["completion_tokens"] += int(meta.get("completion_tokens") or 0)
            if meta.get("spec_accept_length") is not None:
                s["accept_len_sum"] += float(meta["spec_accept_length"]); s["accept_len_n"] += 1
            if err and len(s["errors"]) < 50:
                s["errors"].append(f"{rid}: {err}")
            try:
                with open(self.req_log, "a", encoding="utf-8") as f:
                    f.write(json.dumps(rec) + "\n")
            except Exception:
                pass

    def step(self):
        if not self.server.alive():
            raise RuntimeError(f"SGLang server exited (rc={self.server.proc.returncode}); see {self.server.log_path}")
        items, self.held = list(self.held), []
        if not items:
            try:
                items.append(self.q.get(timeout=0.05))
            except _queue.Empty:
                return []
        while True:
            try:
                items.append(self.q.get_nowait())
            except _queue.Empty:
                break
        merged = {}
        for rid, ids, reason in items:
            m = merged.setdefault(rid, [[], None])
            if m[1] is None:
                m[0].extend(ids); m[1] = reason
        return [_E11Output(rid, ids, reason) for rid, (ids, reason) in merged.items()]

    def take_pending(self, rids):
        """E15P rev 2 (F3, GPT audit 1 HIGH-2): after abort_request() the streams of rids have ended, but their last deltas
        may still sit in the queue (step() moves them only on its next call). Returns rid -> (token ids in order, final
        reason or None) for rids; every other item is kept, in order, for the next step()."""
        rids, out, rest = set(rids), {}, []
        items, self.held = list(self.held), []
        while True:
            try:
                items.append(self.q.get_nowait())
            except _queue.Empty:
                break
        for rid, ids, reason in items:
            if rid in rids:
                o = out.setdefault(rid, [[], None])
                if o[1] is None:
                    o[0].extend(ids)
                    o[1] = reason
            else:
                rest.append((rid, ids, reason))
        self.held = rest
        return {k: (v[0], v[1]) for k, v in out.items()}

    def abort_request(self, ids, wait_sec=30.0):
        """rev 2 (GPT audit, verified: /abort_request only dispatches; scheduler.py:5558-5572 finishes a running request
        after one more decode pass). All POSTs go out in parallel, then this waits (bounded, one shared deadline) until
        each stream has ended, so the slot is released before the caller admits phase 2. Returns rid -> final reason,
        'timeout' or the POST error."""
        ids = list(ids)
        if not ids:
            return {}
        t0, posted = time.perf_counter(), {}
        for rid in ids:
            self.aborted.add(rid)

        def _post(rid):
            try:
                _e11_post("/abort_request", {"rid": rid}, timeout=wait_sec)
                posted[rid] = True
            except Exception as e:
                posted[rid] = f"{type(e).__name__}: {e}"[:200]
        ths = [threading.Thread(target=_post, args=(rid,), daemon=True) for rid in ids]
        for t in ths:
            t.start()
        for t in ths:
            t.join(max(0.0, wait_sec - (time.perf_counter() - t0)))
        while time.perf_counter() - t0 < wait_sec and any(r not in self.done for r in ids):
            time.sleep(0.02)
        out = {}
        with self.lock:
            for rid in ids:
                p = posted.get(rid, "post timeout")
                if p is True:
                    self.stats["aborts_sent"] += 1
                else:
                    self.stats["abort_errors"] += 1
                    if len(self.stats["errors"]) < 50:
                        self.stats["errors"].append(f"abort {rid}: {p}")
                if rid in self.done:
                    out[rid] = self.done[rid]
                else:
                    out[rid] = "timeout" if p is True else p
                    self.stats["abort_timeouts"] += 1
            self.stats["abort_wait_sec_max"] = max(self.stats["abort_wait_sec_max"], round(time.perf_counter() - t0, 2))
        return out


_E11_PROM_RE = re.compile(r"^([A-Za-z_:][A-Za-z0-9_:]*)(\{[^}]*\})?\s+(\S+)")


def e11_prom(timeout=10):
    txt = _urlreq.urlopen(SGL_URL + "/metrics", timeout=timeout).read().decode("utf-8", "replace")
    rows = []
    for line in txt.splitlines():
        if not line or line.startswith("#"):
            continue
        m = _E11_PROM_RE.match(line)
        if not m:
            continue
        try:
            rows.append(_types.SimpleNamespace(name=m.group(1), labels=m.group(2), value=float(m.group(3))))
        except ValueError:
            continue
    return rows


class E11SglangLLM:
    """llm.get_metrics() for the S49 KV gate: the gate reads names containing 'kv_cache_usage'. SGLang's full_token_usage
    (KV pool used, evictable cache excluded) is given that name; token_usage = max(full, mamba) is kept as it is."""

    def get_metrics(self):
        rows = e11_prom()
        # E15P rev 2 (F4, GPT audits 1-2): the bottleneck pool, not the KV pool alone. SGLang 0.5.21 token_usage =
        # max(full, swa, mamba) (pool_stats_observer.py:69-73, 128); mamba_usage counts active slots only (:272-276)
        anyu = [r.value for r in rows if r.name == "sglang:token_usage"]
        pools = [r.value for r in rows if r.name in ("sglang:full_token_usage", "sglang:mamba_usage")]
        kv = max(anyu) if anyu else (max(pools) if pools else None)
        if kv is not None and kv == kv and 0.0 <= kv <= 1.5:
            rows.append(_types.SimpleNamespace(name="e11:kv_cache_usage(sglang:token_usage=max(full,mamba))", labels=None,
                                               value=kv))
        return rows


_E11_SNAP_KEYS = ("num_running_reqs", "num_queue_reqs", "token_usage", "gen_throughput", "spec_accept", "cache_hit_rate",
                  "retract", "generation_tokens_total", "prompt_tokens_total", "cached_tokens_total", "kv_cache_usage",
                  "num_requests_total", "mamba_usage")   # E15P rev 2 (F12): + mamba_usage


def metrics_snapshot():
    if not ENGINE_READY:
        return []
    try:
        return [{"name": r.name, "labels": r.labels, "value": r.value} for r in llm.get_metrics()
                if any(k in r.name for k in _E11_SNAP_KEYS) and not r.name.endswith(("_bucket", "_created"))]
    except Exception as e:
        return [{"error": repr(e)[:300]}]


_E11_KV_RE = re.compile(r"([A-Za-z#][A-Za-z#_ ()/\-]*?):\s*(-?\d+(?:\.\d+)?)")


def e11_decode_summary():
    """Decode rate by running level and by mean context per running request, from the server's TP0 decode lines."""
    if E11_SERVER is None:
        return None
    rows = []
    _texts = []
    for _p in list(globals().get("E15P_OLD_LOGS") or []):     # E15P rev 2: servers replaced by a restart
        try:
            _texts.append(Path(_p).read_text(encoding="utf-8", errors="replace"))
        except Exception:
            pass
    _texts.append(E11_SERVER.log_text())
    for line in "\n".join(_texts).splitlines():
        if "Decode batch" in line and "TP0]" in line:
            rows.append({k.strip(): float(v) for k, v in _E11_KV_RE.findall(line.split("Decode batch", 1)[1])})
    def med(v):
        return round(_statistics.median(v), 1) if v else None
    bins = {}
    for r in rows:
        n, ft, g = r.get("#running-req"), r.get("#full token"), r.get("gen throughput (token/s)")
        if not n or ft is None or g is None:
            continue
        ctx = ft / n
        k = "<10k" if ctx < 10000 else "10-20k" if ctx < 20000 else "20-40k" if ctx < 40000 else "40-60k" if ctx < 60000 else ">=60k"
        bins.setdefault(f"running={int(n)}" if n < SGL_MAX_RUNNING else "running=max", {}).setdefault(k, []).append(g)
    out = {"n_lines": len(rows),
           "running_hist": {str(int(k)): sum(1 for r in rows if r.get("#running-req") == k) for k in sorted({r.get("#running-req", 0) for r in rows})},
           "gen_tok_s_median_at_max": med([r["gen throughput (token/s)"] for r in rows
                                           if r.get("#running-req") == SGL_MAX_RUNNING and "gen throughput (token/s)" in r]),
           "accept_len_median": med([r["accept len"] for r in rows if "accept len" in r]),
           "queue_max": max((r.get("#queue-req", 0) for r in rows), default=None),
           "retract_lines": sum(1 for l in "\n".join(_texts).splitlines() if "retract" in l.lower() and "retraction_policy" not in l and "server_args" not in l.lower()),
           # E15P rev 2 (F12): pool maxima seen in the decode lines
           "full_token_usage_max": max((r["full token usage"] for r in rows if "full token usage" in r), default=None),
           "mamba_usage_max": max((r["mamba usage"] for r in rows if "mamba usage" in r), default=None),
           "running_max": max((r.get("#running-req", 0) for r in rows), default=None),
           "by_context": {lvl: {k: {"lines": len(v), "gen_tok_s_median": med(v)} for k, v in d.items()} for lvl, d in bins.items()}}
    try:
        write_json_atomic(OUT / "sglang_decode_summary.json", out)
    except Exception:
        pass
    return out


def e11_smoke(eng):
    """The shim's own path: (a) greedy 16 tokens, streamed deltas == completion_tokens; (b) abort mid-stream -> the stream
    ends with finish 'abort' (the drain and the hard stop rely on it to free the slot)."""
    res, t0 = {}, time.perf_counter()
    ids = tokenizer.encode("Reply with the digits 1 2 3 4 5 6 7 8:", add_special_tokens=False)
    eng.add_request("e11-smoke-a", {"prompt_token_ids": ids}, SamplingParams(max_tokens=16, temperature=0.0, top_p=1.0, top_k=-1))
    got, fin = [], None
    while fin is None and time.perf_counter() - t0 < 300:
        for o in eng.step():
            if o.request_id == "e11-smoke-a":
                got += o.outputs[0].token_ids; fin = o.outputs[0].finish_reason if o.finished else None
    time.sleep(0.5)
    logged = [json.loads(l) for l in open(eng.req_log, encoding="utf-8")] if eng.req_log.exists() else []
    meta = next((r for r in logged if r["rid"] == "e11-smoke-a"), {})
    res["a"] = {"tokens": len(got), "finish": fin, "completion_tokens": meta.get("completion_tokens"),
                "text": tokenizer.decode(got)[:80]}
    res["a"]["ok"] = bool(got) and fin in ("stop", "length") and meta.get("completion_tokens") == len(got)
    t1 = time.perf_counter()
    eng.add_request("e11-smoke-b", {"prompt_token_ids": ids}, SamplingParams(max_tokens=4096, temperature=0.6, top_p=0.95, top_k=20))
    got, fin, sent = [], None, False
    while fin is None and time.perf_counter() - t1 < 300:
        for o in eng.step():
            if o.request_id == "e11-smoke-b":
                got += o.outputs[0].token_ids; fin = o.outputs[0].finish_reason if o.finished else None
        if got and not sent:
            eng.abort_request(["e11-smoke-b"]); sent = True; t_abort = time.perf_counter()
        if sent and time.perf_counter() - t_abort > 60:
            break
    res["b"] = {"tokens": len(got), "finish": fin, "abort_sent": sent,
                "abort_to_finish_sec": round(time.perf_counter() - t_abort, 2) if sent else None}
    res["b"]["ok"] = sent and fin == "abort" and len(got) < 4096
    res["ok"] = res["a"]["ok"] and res["b"]["ok"]
    res["sec"] = round(time.perf_counter() - t0, 1)
    return res


def e11_bring_up():
    global E11_SERVER, SGL_MAX_RUNNING, SGL_SERVER_MAX_RUNNING   # E15B: the 12-stream fallbacks
    E11_SGL["stage"] = "inputs"
    inp = {"int4": SGL_MODEL_PATH, "dflash2": SGL_DRAFT_PATH, "wheelhouse": SGL_WHEELHOUSE}
    E11_SGL["inputs"] = {k: str(v) for k, v in inp.items()}
    missing = [k for k, v in inp.items() if not (v / ("BUILD_INFO.json" if k == "wheelhouse" else "config.json")).is_file()]
    if missing:
        raise E11Fail(f"inputs missing: {missing}")
    if (_load_json(SGL_MODEL_PATH / "config.json").get("quantization_config") or {}).get("quant_method") != "compressed-tensors":
        raise E11Fail("INT4 checkpoint is not compressed-tensors")
    E11_SGL["stage"] = "install"
    vpy, env, E11_SGL["install"] = e11_install()
    launch = [str(vpy), "-m", "sglang.launch_server"] if not E11_HARNESS else [sys.executable, os.environ["E11_FAKE_SERVER"]]
    e14_after_install(vpy, env)             # E14: NCCL bench while the copy runs, then the staged model paths
    E14_VENV["launch"] = list(launch)
    cfg, used = dict(W1), []
    for i in range(4):      # E15B B5: 16 -> 12 -> diagnosed -> W1
        left = SGL_GIVE_UP_AT - time.perf_counter()
        if left < 60:
            raise E11Fail("give-up deadline reached before a launch")
        E11_SGL["stage"] = f"launch {i}"
        srv = E11Server(launch + e11_flags(cfg), env, f"try{i}")
        srv.start()
        # E14-0: the first launch may use the time up to the give-up (the stall rule decides), relaunches keep 20 min
        ok, why = srv.wait_ready(left if i == 0 else min(SGL_RELAUNCH_TIMEOUT_SEC, left))
        att = {"try": i, "cfg": dict(cfg), "ready": ok, "why": why, "sec": round(time.perf_counter() - srv.t0, 1)}
        E11_SGL["attempts"].append(att)
        print(f"E11 SGLang launch {i} {cfg}: {why}", flush=True)
        if ok:
            E11_SERVER = srv
            break
        tail = srv.log_text()[-20000:]
        att["log_tail"] = tail[-3000:]
        srv.kill()
        att["gpu_free"] = _e11_gpu_free()
        nxt = e11_next(cfg, tail, used)
        if nxt is None:
            raise E11Fail(f"launch {i} failed ({why}); no diagnosed fallback")
        cfg, reason = nxt
        used.append(reason); att["retry_reason"] = reason
    else:
        raise E11Fail("4 launches failed")
    E11_SGL["launch_flags"] = e11_flags(cfg)
    E14_VENV["cfg"] = dict(cfg)
    E14_STATE.enter("READY", ready=getattr(E11_SERVER, "e14_stats", None))
    caps = e11_caps(E11_SERVER.log_text())
    caps["gate"], caps["concurrency"] = e11_capacity_gate(caps, cfg)
    if not caps["gate"]["pass"] and not cfg.get("streams12") and SGL_GIVE_UP_AT - time.perf_counter() > 120:
        # E15B #16: the 16-stream server came up but failed the capacity gate (e.g. GPU headroom): one relaunch at the
        # E14S 12-stream size, the delayer still off
        E11_SGL["gate16"] = {"caps": dict(caps), "cfg": dict(cfg)}
        print("E15B: 16-stream capacity gate failed -> 12-stream relaunch:", json.dumps(caps["gate"]), flush=True)
        E11_SERVER.kill()
        _e11_gpu_free()
        SGL_MAX_RUNNING, SGL_SERVER_MAX_RUNNING = 12, 13
        cfg = dict(cfg, streams12=True, mamba_cache=56)
        srv = E11Server(launch + e11_flags(cfg), env, "try_s12")
        srv.start()
        ok, why = srv.wait_ready(min(SGL_RELAUNCH_TIMEOUT_SEC, max(60.0, SGL_GIVE_UP_AT - time.perf_counter())))
        E11_SGL["attempts"].append({"try": "s12", "cfg": dict(cfg), "ready": ok, "why": why,
                                    "sec": round(time.perf_counter() - srv.t0, 1)})
        if not ok:
            srv.kill()
            raise E11Fail(f"12-stream relaunch after the 16-stream gate failed: {why}")
        E11_SERVER = srv
        E11_SGL["launch_flags"] = e11_flags(cfg)
        E14_VENV["cfg"] = dict(cfg)
        caps = e11_caps(E11_SERVER.log_text())
        caps["gate"], caps["concurrency"] = e11_capacity_gate(caps, cfg)
    E11_SGL["caps"] = caps
    print("E11 caps:", json.dumps(caps), flush=True)
    if not caps["gate"]["pass"]:
        raise E11Fail("capacity gate failed: " + json.dumps(caps["gate"]))
    E11_SGL["stage"] = "smoke"
    eng = E11SglangEngine(E11_SERVER)
    E11_SGL["smoke"] = e11_smoke(eng)
    print("E11 smoke:", json.dumps(E11_SGL["smoke"]), flush=True)
    if not E11_SGL["smoke"]["ok"]:
        if not cfg.get("streams12") and SGL_GIVE_UP_AT - time.perf_counter() > 120:   # E15P rev 2 (F15, GPT audit 2)
            E11_SGL["smoke16"] = dict(E11_SGL["smoke"])
            print("E15P: 16-stream smoke failed -> 12-stream relaunch:", json.dumps(E11_SGL["smoke"])[:600], flush=True)
            return e15p_relaunch_s12("smoke", min(SGL_RELAUNCH_TIMEOUT_SEC, max(60.0, SGL_GIVE_UP_AT - time.perf_counter())))
        raise E11Fail("smoke failed: " + json.dumps(E11_SGL["smoke"])[:600])
    return eng, caps


E15P_OLD_LOGS = []          # server logs of servers replaced by a relaunch (F12 decode summary)
E15P_RELAUNCHES = []


def e15p_relaunch_s12(reason, timeout_sec):
    """E15P rev 2 (F2 / F15): stop the current server and start the 12-stream configuration (mrr 13, graphs 1..13, mamba
    56; the W1 flags stay W1) with the same venv, env and staged model (no reinstall); then the capacity gate and the
    smoke. Returns (engine, caps); raises E11Fail. Sets SGL_MAX_RUNNING 12 so the gate gives concurrency 12."""
    global E11_SERVER, SGL_MAX_RUNNING, SGL_SERVER_MAX_RUNNING
    rec = {"reason": reason, "t": round(time.perf_counter() - NOTEBOOK_T0), "timeout_sec": round(timeout_sec)}
    E15P_RELAUNCHES.append(rec)
    old = E11_SERVER
    env = getattr(old, "env", None)
    if old is not None:
        try:
            E15P_OLD_LOGS.append(str(old.log_path))
            old.kill()
        except Exception as e:
            rec["kill_error"] = repr(e)[:200]
    rec["gpu_free"] = _e11_gpu_free()
    if env is None or not E14_VENV.get("launch"):
        raise E11Fail("relaunch: no launch command / env recorded")
    cfg0 = dict(E14_VENV.get("cfg") or W1)
    cfg = dict(cfg0) if (cfg0.get("w1") or cfg0.get("streams12")) else dict(cfg0, streams12=True, mamba_cache=56)
    SGL_MAX_RUNNING, SGL_SERVER_MAX_RUNNING = 12, 13
    srv = E11Server(list(E14_VENV["launch"]) + e11_flags(cfg), env, f"relaunch{len(E15P_RELAUNCHES)}_{reason}")
    srv.start()
    ok, why = srv.wait_ready(timeout_sec)
    rec.update(ready=ok, why=why, cfg=dict(cfg), sec=round(time.perf_counter() - srv.t0, 1))
    E11_SGL.setdefault("attempts", []).append(dict(rec, **{"try": "relaunch_" + reason}))
    if not ok:
        srv.kill()
        raise E11Fail(f"12-stream relaunch ({reason}) not ready: {why}")
    E11_SERVER = srv
    E11_SGL["launch_flags"] = e11_flags(cfg)
    E14_VENV["cfg"] = dict(cfg)
    caps = e11_caps(srv.log_text())
    caps["gate"], caps["concurrency"] = e11_capacity_gate(caps, cfg)
    rec["gate"] = caps["gate"]
    if not caps["gate"]["pass"]:
        raise E11Fail(f"12-stream relaunch ({reason}) capacity gate failed: " + json.dumps(caps["gate"]))
    eng_ = E11SglangEngine(srv)
    E11_SGL["smoke"] = e11_smoke(eng_)
    rec["smoke_ok"] = bool(E11_SGL["smoke"]["ok"])
    if not E11_SGL["smoke"]["ok"]:
        raise E11Fail(f"12-stream relaunch ({reason}) smoke failed: " + json.dumps(E11_SGL["smoke"])[:400])
    E11_SGL["caps"] = caps
    print("E15P relaunch:", json.dumps(rec, default=str)[:800], flush=True)
    return eng_, caps


_t_engine = time.perf_counter()
if not E10_SANDBOX_SELFTEST.get("ok", False):
    E11_SGL["stage"] = "skipped: sandbox self-test failed"
elif ENGINE_BACKEND != "sglang":
    E11_SGL["stage"] = f"skipped: ENGINE_BACKEND={ENGINE_BACKEND}"
else:
    try:
        engine, _caps = e11_bring_up()
        llm = E11SglangLLM()
        ENGINE_READY = True
        CONCURRENCY = int(_caps["concurrency"])                     # rev 2: from the capacity gate (W1: 12)
        # rev 2: SGLang-only clock and KV gate (see CELL 1): drain 12 min before the hard stop; gate from 10 running
        if "drain_margin_sec" in P and P["drain_margin_sec"] < SGL_DRAIN_MARGIN_SEC:
            DRAIN_DEADLINE = HARD_STOP - SGL_DRAIN_MARGIN_SEC
            ADMIT_DEADLINE = DRAIN_DEADLINE - P["admit_margin_sec"]
            assert NOTEBOOK_T0 < ADMIT_DEADLINE < DRAIN_DEADLINE < HARD_STOP
        # rev 3b (user 2026-10-18): request length limit = the server's context (155,648, the E11-04 launch), so the
        # 60k thinking floor never forces a feedback trim or a skip (room for prompts up to ~87k; real max seen 28k).
        # phase1_cap() and phase 2 read MAX_MODEL_LEN at call time; the vLLM fallback keeps S1-60K's 106,496.
        MAX_MODEL_LEN = SGL_CONTEXT_LEN
        PROMPT_TOKEN_LIMIT = MAX_MODEL_LEN - ANSWER_MAX_TOKENS - CONTEXT_SAFETY - MIN_THINK_BUDGET
        assert phase1_cap(38049) == THINK_BUDGET and phase1_cap(80000) == THINK_BUDGET
        E10_KV_GATE_MIN_ACTIVE = min(E10_KV_GATE_MIN_ACTIVE, SGL_KV_GATE_MIN_ACTIVE)
        E10_KV_ADMIT_MAX = min(E10_KV_ADMIT_MAX, SGL_KV_ADMIT_MAX)
        E11_SGL.update(ok=True, stage="ready")
        ENGINE_INFO.update(backend="sglang-0.5.21-int4-dflash2-" + ("w1" if (E14_VENV.get("cfg") or {}).get("w1") else "w6") + f"-s{SGL_MAX_RUNNING}", kv_tokens=_caps.get("kv_tokens"),
                           effective_running=_caps.get("effective_running"),
                           drain_before_hard_stop_sec=round(HARD_STOP - DRAIN_DEADLINE), max_model_len=MAX_MODEL_LEN,
                           kv_gate={"min_active": E10_KV_GATE_MIN_ACTIVE, "admit_max": E10_KV_ADMIT_MAX})
    except Exception as _e:
        E11_SGL["error"] = f"{type(_e).__name__}: {_e}"[:3000]
        E11_SGL["traceback"] = traceback.format_exc()[-3000:]
        print("E11 SGLang engine NOT ready:", E11_SGL["stage"], "|", E11_SGL["error"][:1500], flush=True)
        e11_kill_server(); E11_SERVER = None
        engine = None; llm = None; ENGINE_READY = False; CONCURRENCY = 0
ENGINE_INFO["load_sec"] = round(time.perf_counter() - _t_engine, 1)
ENGINE_INFO["concurrency"] = int(CONCURRENCY)
print("engine:", json.dumps(ENGINE_INFO, default=str)[:3000], flush=True)


In [ ]:
# ==================================================================== E17 hook source (src/e17_hook.py)
# Written to /tmp/e17_hook/sitecustomize.py and put on PYTHONPATH of the E17 server launches only (inert without
# E17_TRACE_DIR); it wraps four SGLang methods in place and never edits a source file.
E17_HOOK_SHA = "27119b7a1625f924132296dad3a737c93fb18d3f0aa283f03c5e59b0db255e6c"
E17_HOOK_SRC = "\"\"\"E17 instrumentation for SGLang 0.5.21, the #43567 stale lazy-checkpoint chain (PLAN_E17 rev 2 \u00a72).\n\nInstalled as `sitecustomize.py` in a directory put on PYTHONPATH of the server launch only. It is inert unless\nE17_TRACE_DIR is set. It never edits a source file: a daemon thread waits until the target modules are imported, then\nwraps four methods in place. Each wrapper infers the outcome from the request state before and after the original\ncall. The unpatched arm therefore runs byte-identical SGLang code.\n\nWrapped (0.5.21 line numbers):\n- ScheduleBatch.mamba_lazy_prealloc_at_boundary (schedule_batch.py:3464-3493). Eligible = seq_len % interval == 0\n  and no pending slot. Afterwards the pending slot is still -1 -> optional_alloc_failed; the request is then marked\n  `inplace_pending` (the forward writes the new state into the keep slot).\n- ScheduleBatch.mamba_lazy_spec_prepare (:3495-3533). In window and no pending before; the planned position equals the\n  keep index afterwards -> spec_inplace_fallback (same mark).\n- SchedulerBatchResultProcessor._mamba_prefix_cache_update / _mamba_lazy_spec_update (batch_result_processor.py:\n  1381-1490). A new non-None mamba_last_track_seqlen clears the mark (the checkpoint is consistent again).\n- MambaComponent.prepare_for_caching_req (unified_cache/components/mamba.py:539). Logs every donation.\n  stale_donation_candidate = marked and the length is not None and the returned length is > 0.\n- ModelRunner.load_model (TP rank 0): logs the KV-cache scales of every attention layer once.\n\nOutput, per process: trace_<pid>.jsonl (events, appended) and counters_<pid>.json (counters, the list of patched\ntargets and an allocator snapshot), both flushed every E17_FLUSH_SEC (default 2 s) by a daemon thread. Nothing is\nwritten on the scheduler's hot path, and there is no CUDA sync beyond the .item() reads the original code does for the\nsame requests.\n\"\"\"\nimport json\nimport os\nimport sys\nimport threading\nimport time\nfrom collections import deque\n\n_DIR = os.environ.get(\"E17_TRACE_DIR\")\n_VERBOSE = os.environ.get(\"E17_TRACE_VERBOSE\") == \"1\"\n_FLUSH = float(os.environ.get(\"E17_FLUSH_SEC\", \"2\"))\n_PID = os.getpid()\n_LOCK = threading.Lock()\nEVENTS = deque(maxlen=200000)\nCOUNTERS = {}\nPATCHED = []\nMARK = {}             # rid -> {\"since\": seq_len, \"len\": mamba_last_track_seqlen at the failure, \"path\": ...}\nPEND = {}             # rid -> a spec-path stale candidate waiting for its verify result (crossed or not)\n_REFS = {}            # the last pool / tree cache seen (allocator snapshot)\n_SEQ = [0]\nTARGETS = {\n    \"sglang.srt.managers.schedule_batch\": \"ScheduleBatch\",\n    \"sglang.srt.managers.scheduler_components.batch_result_processor\": \"SchedulerBatchResultProcessor\",\n    \"sglang.srt.mem_cache.unified_cache.components.mamba\": \"MambaComponent\",\n    \"sglang.srt.model_executor.model_runner\": \"ModelRunner\",\n}\n\n\ndef _count(k, n=1):\n    COUNTERS[k] = COUNTERS.get(k, 0) + n\n\n\ndef _ev(kind, **kw):\n    _SEQ[0] += 1\n    kw.update(ev=kind, i=_SEQ[0], t=round(time.time(), 3))\n    EVENTS.append(kw)\n\n\ndef _item(x):\n    try:\n        return int(x.item())\n    except Exception:\n        try:\n            return int(x)\n        except Exception:\n            return None\n\n\ndef _rid(req):\n    return str(getattr(req, \"rid\", \"\"))\n\n\ndef _kv(req):\n    return getattr(req, \"kv\", None)\n\n\ndef _wrap_prealloc(cls):\n    orig = cls.mamba_lazy_prealloc_at_boundary\n\n    def mamba_lazy_prealloc_at_boundary(self, mamba_track_interval, *a, **k):\n        pre = []\n        try:\n            _REFS[\"pool\"], _REFS[\"tree\"] = self.req_to_token_pool, self.tree_cache\n            for i, req in enumerate(self.reqs):\n                kv = _kv(req)\n                buf = getattr(kv, \"mamba_ping_pong_track_buffer\", None)\n                if buf is None or _item(self.seq_lens_cpu[i]) % mamba_track_interval != 0:\n                    continue\n                other = 1 - kv.mamba_next_track_idx\n                if _item(buf[other]) != -1:\n                    _count(\"boundary_skip_pending\")\n                    continue\n                pre.append((i, req, other, kv.mamba_last_track_seqlen, _item(self.seq_lens_cpu[i])))\n        except Exception as e:\n            _count(\"hook_error_prealloc_pre\")\n            _ev(\"hook_error\", where=\"prealloc_pre\", err=repr(e)[:200])\n        out = orig(self, mamba_track_interval, *a, **k)\n        try:\n            for i, req, other, last, seq in pre:\n                _count(\"optional_alloc_attempts\")\n                kv = _kv(req)\n                if _item(kv.mamba_ping_pong_track_buffer[other]) == -1:\n                    _count(\"optional_alloc_failed\")\n                    MARK[_rid(req)] = {\"since\": seq, \"len\": last, \"path\": \"decode\"}\n                    _ev(\"alloc_failed\", path=\"decode\", rid=_rid(req), seq_len=seq, last_len_before=last,\n                        last_len_after=kv.mamba_last_track_seqlen, interval=mamba_track_interval,\n                        keep_idx=kv.mamba_next_track_idx,\n                        available=_item(self.req_to_token_pool.mamba_allocator.available_size()))\n                elif _VERBOSE:\n                    _ev(\"alloc_ok\", path=\"decode\", rid=_rid(req), seq_len=seq)\n        except Exception as e:\n            _count(\"hook_error_prealloc_post\")\n            _ev(\"hook_error\", where=\"prealloc_post\", err=repr(e)[:200])\n        return out\n\n    cls.mamba_lazy_prealloc_at_boundary = mamba_lazy_prealloc_at_boundary\n\n\ndef _wrap_spec_prepare(cls, in_window):\n    orig = cls.mamba_lazy_spec_prepare\n\n    def mamba_lazy_spec_prepare(self, mamba_track_interval, max_draft_tokens, *a, **k):\n        pre = []\n        try:\n            _REFS[\"pool\"], _REFS[\"tree\"] = self.req_to_token_pool, self.tree_cache\n            for i, req in enumerate(self.reqs):\n                kv = _kv(req)\n                buf = getattr(kv, \"mamba_ping_pong_track_buffer\", None)\n                if buf is None or not in_window(req, mamba_track_interval, max_draft_tokens):\n                    continue\n                other = 1 - kv.mamba_next_track_idx\n                pre.append((i, req, _item(buf[other]) != -1, kv.mamba_last_track_seqlen, kv.kv_committed_len))\n        except Exception as e:\n            _count(\"hook_error_spec_pre\")\n            _ev(\"hook_error\", where=\"spec_pre\", err=repr(e)[:200])\n        out = orig(self, mamba_track_interval, max_draft_tokens, *a, **k)\n        try:\n            pos = getattr(self, \"mamba_lazy_spec_track_positions_cpu\", None) or []\n            for i, req, had_pending, last, committed in pre:\n                _count(\"spec_window_reachable\")\n                if had_pending:\n                    continue\n                _count(\"optional_alloc_attempts\")\n                kv = _kv(req)\n                if i < len(pos) and pos[i] == kv.mamba_next_track_idx:\n                    _count(\"optional_alloc_failed\")\n                    _count(\"spec_inplace_fallback\")\n                    MARK[_rid(req)] = {\"since\": committed, \"len\": last, \"path\": \"spec\"}\n                    _ev(\"alloc_failed\", path=\"spec\", rid=_rid(req), kv_committed_len=committed, last_len_before=last,\n                        last_len_after=kv.mamba_last_track_seqlen, interval=mamba_track_interval,\n                        max_draft_tokens=max_draft_tokens, keep_idx=kv.mamba_next_track_idx,\n                        available=_item(self.req_to_token_pool.mamba_allocator.available_size()))\n                elif _VERBOSE:\n                    _ev(\"alloc_ok\", path=\"spec\", rid=_rid(req), kv_committed_len=committed)\n        except Exception as e:\n            _count(\"hook_error_spec_post\")\n            _ev(\"hook_error\", where=\"spec_post\", err=repr(e)[:200])\n        return out\n\n    cls.mamba_lazy_spec_prepare = mamba_lazy_spec_prepare\n\n\ndef _wrap_update(cls, name, spec):\n    orig = getattr(cls, name)\n\n    def wrapped(self, req, batch, *a, **k):\n        kv = _kv(req)\n        before = getattr(kv, \"mamba_last_track_seqlen\", None)\n        out = orig(self, req, batch, *a, **k)\n        try:\n            after = getattr(kv, \"mamba_last_track_seqlen\", None)\n            if spec:\n                # the verify that a spec fallback planned has now been processed: a real crossing wrote the keep slot in\n                # place (a pending stale candidate is confirmed); no crossing wrote nothing (the candidate was harmless)\n                crossed = a[1] if len(a) > 1 else k.get(\"crossed\")\n                if crossed:\n                    _count(\"spec_crossed\")\n                m0 = MARK.get(_rid(req))\n                if m0 is not None and m0.get(\"path\") == \"spec\":\n                    if PEND.pop(_rid(req), None):\n                        _count(\"stale_donation_confirmed\" if crossed else \"stale_candidate_cleared\")\n                        _ev(\"spec_resolve\", rid=_rid(req), crossed=bool(crossed), confirmed=bool(crossed))\n                    if not crossed:\n                        MARK.pop(_rid(req), None)\n            if after is not None and after != before:\n                _count(\"track_len_updates\")\n                m = MARK.pop(_rid(req), None)\n                if m is not None or _VERBOSE:\n                    _ev(\"track_len_update\", rid=_rid(req), before=before, after=after, was_marked=m is not None,\n                        spec=spec, crossed=(a[1] if spec and len(a) > 1 else None))\n        except Exception as e:\n            _count(\"hook_error_update\")\n            _ev(\"hook_error\", where=name, err=repr(e)[:200])\n        return out\n\n    wrapped.__name__ = name\n    setattr(cls, name, wrapped)\n\n\ndef _wrap_donate(cls):\n    orig = cls.prepare_for_caching_req\n\n    def prepare_for_caching_req(self, req, insert_params, token_ids_len, is_finished, *a, **k):\n        kv = _kv(req)\n        try:\n            extra = bool(getattr(self.cache, \"enable_mamba_extra_buffer\", False))\n        except Exception:\n            extra = None\n        cache_len = getattr(kv, \"mamba_last_track_seqlen\", None) if extra else token_ids_len\n        m = MARK.get(_rid(req))\n        out = orig(self, req, insert_params, token_ids_len, is_finished, *a, **k)\n        try:\n            _count(\"donations_finished\" if is_finished else \"donations_unfinished\")\n            stale = bool(m is not None and extra and cache_len is not None and isinstance(out, int) and out > 0)\n            if stale:\n                _count(\"stale_donation_candidate\")\n                if m.get(\"path\") == \"decode\":       # the boundary forward writes the keep slot for certain\n                    _count(\"stale_donation_confirmed\")\n                else:\n                    PEND[_rid(req)] = True\n            if m is not None:\n                _count(\"donations_while_marked\")\n            if stale or m is not None or _VERBOSE:\n                _ev(\"donation\", rid=_rid(req), finished=bool(is_finished), cache_len=cache_len, returned=out,\n                    token_ids_len=token_ids_len, marked=m, stale_candidate=stale, extra_buffer=extra)\n            if is_finished:\n                MARK.pop(_rid(req), None)\n                PEND.pop(_rid(req), None)\n        except Exception as e:\n            _count(\"hook_error_donate\")\n            _ev(\"hook_error\", where=\"donate\", err=repr(e)[:200])\n        return out\n\n    cls.prepare_for_caching_req = prepare_for_caching_req\n\n\ndef _wrap_load(cls):\n    orig = cls.load_model\n\n    def load_model(self, *a, **k):\n        out = orig(self, *a, **k)\n        try:\n            if getattr(self, \"tp_rank\", 0) == 0 and not getattr(self, \"is_draft_worker\", False):\n                rows = []\n                for name, mod in self.model.named_modules():\n                    ks, vs = getattr(mod, \"k_scale_float\", None), getattr(mod, \"v_scale_float\", None)\n                    if ks is None and vs is None and not hasattr(mod, \"k_scale\"):\n                        continue\n                    rows.append({\"layer\": name, \"k_scale_float\": ks, \"v_scale_float\": vs,\n                                 \"k_scale\": _f(getattr(mod, \"k_scale\", None)), \"v_scale\": _f(getattr(mod, \"v_scale\", None))})\n                _ev(\"kv_scales\", n=len(rows), rows=rows[:128])\n        except Exception as e:\n            _ev(\"hook_error\", where=\"load_model\", err=repr(e)[:200])\n        return out\n\n    cls.load_model = load_model\n\n\ndef _f(x):\n    try:\n        return float(x.item() if hasattr(x, \"item\") and getattr(x, \"numel\", lambda: 1)() == 1 else x)\n    except Exception:\n        return None if x is None else str(type(x).__name__)\n\n\ndef _try_patch():\n    left = [m for m in TARGETS if TARGETS[m] + \"@\" + m not in PATCHED]\n    for mod in left:\n        m = sys.modules.get(mod)\n        if m is None or not hasattr(m, TARGETS[mod]):\n            continue\n        cls = getattr(m, TARGETS[mod])\n        try:\n            if mod.endswith(\"schedule_batch\"):\n                _wrap_prealloc(cls)\n                _wrap_spec_prepare(cls, getattr(m, \"mamba_lazy_spec_in_window\"))\n            elif mod.endswith(\"batch_result_processor\"):\n                _wrap_update(cls, \"_mamba_prefix_cache_update\", spec=False)\n                _wrap_update(cls, \"_mamba_lazy_spec_update\", spec=True)\n            elif mod.endswith(\"components.mamba\"):\n                _wrap_donate(cls)\n            else:\n                _wrap_load(cls)\n            PATCHED.append(TARGETS[mod] + \"@\" + mod)\n            _ev(\"patched\", target=TARGETS[mod], module=mod)\n        except Exception as e:\n            PATCHED.append(TARGETS[mod] + \"@\" + mod)\n            _ev(\"patch_failed\", target=TARGETS[mod], module=mod, err=repr(e)[:300])\n            _count(\"patch_failed\")\n    return len(PATCHED) == len(TARGETS)\n\n\ndef _snapshot():\n    snap = {}\n    try:\n        pool = _REFS.get(\"pool\")\n        if pool is not None:\n            snap[\"available\"] = _item(pool.mamba_allocator.available_size())\n            snap[\"size\"] = _item(getattr(getattr(pool, \"mamba_pool\", None), \"size\", None))\n        tree = _REFS.get(\"tree\")\n        if tree is not None and hasattr(tree, \"mamba_evictable_size\"):\n            snap[\"evictable\"] = _item(tree.mamba_evictable_size())\n    except Exception as e:\n        snap[\"error\"] = repr(e)[:200]\n    return snap\n\n\ndef _flush():\n    os.makedirs(_DIR, exist_ok=True)\n    evs = []\n    while EVENTS:\n        evs.append(EVENTS.popleft())\n    if evs:\n        with open(os.path.join(_DIR, f\"trace_{os.getpid()}.jsonl\"), \"a\", encoding=\"utf-8\") as f:\n            for e in evs:\n                f.write(json.dumps(e, default=str) + \"\\n\")\n    if PATCHED or COUNTERS:\n        p = os.path.join(_DIR, f\"counters_{os.getpid()}.json\")\n        tmp = p + \".tmp\"\n        with open(tmp, \"w\", encoding=\"utf-8\") as f:\n            json.dump({\"pid\": os.getpid(), \"argv0\": (sys.argv or [\"\"])[0][-120:], \"patched\": PATCHED, \"counters\": COUNTERS,\n                       \"marked_now\": len(MARK), \"alloc\": _snapshot(), \"t\": round(time.time(), 3)}, f, default=str)\n        os.replace(tmp, p)\n\n\ndef _loop():\n    t_end = time.time() + 3600\n    done = False\n    while True:\n        try:\n            if not done:\n                done = _try_patch() or time.time() > t_end\n            _flush()\n        except Exception:\n            pass\n        time.sleep(_FLUSH if done else 0.05)\n\n\ndef _start():\n    threading.Thread(target=_loop, name=\"e17-hook\", daemon=True).start()\n\n\ndef _after_fork():\n    # a forked child inherits the patched classes and the buffers but not the thread; its files carry its own pid\n    EVENTS.clear()\n    COUNTERS.clear()\n    MARK.clear()\n    PEND.clear()\n    _start()\n\n\nif _DIR:\n    _start()\n    if hasattr(os, \"register_at_fork\"):\n        os.register_at_fork(after_in_child=_after_fork)\n"
assert hashlib.sha256(E17_HOOK_SRC.encode("utf-8")).hexdigest() == E17_HOOK_SHA


In [ ]:
# ==================================================================== CELL E17 — SGLang silent-accuracy tests (PLAN_E17 rev 2)
# One session, engine only, every server TP4 with the production numerics (RedHat INT4, FP8 KV, fp16 SSM,
# extra_buffer_lazy, overlap, decode CUDA graphs). The production launch made by the engine cell is the first launch (JIT
# warm, Phase 0 facts); every arm below is its own relaunch (71-97 s measured in EMS RISKY v1).
#   Phase 0      provenance + safety asserts (schedule_batch sha, #43567 state, DFlash taps, vocab, GDN backend)
#   L1-L6        unpatched: T1a small-slot upstream repro (interval 128, hook on / off, interval 256), T1b forced fail,
#                T1c DFlash W6 (interval 64), L6 = upstream Class 2 (no_buffer + MTP, #43387 known bug: harness check)
#   L12-L13      T3b: target-only vs DFlash, radix cache off, serial 4 + 16 concurrent, greedy 2,048 (T3a fp16 scores)
#   patch #43567 L7-L9: the T1 arms again (+ the patch-safety cases on L7)
#   decision A   BUG_EXPOSED_43567 -> T2 natural pressure (production flags, 16 streams), patched L11, revert, unpatched L10
#   L15          T3a fp32 SSM (only if time is left)
# Instrumentation: e17_hook (sitecustomize on PYTHONPATH of the server only; no SGLang source edit). Verdict ceiling:
# ENGINE_CORRECTNESS_VALIDATED. Every phase has its own deadline; whatever does not fit is skipped and recorded.
import random as _e17_rnd
import re
import shutil
import subprocess
import threading
import traceback
import shlex as _e17_shlex
import statistics as _e17_st
import numpy as np

E17 = {"plan": "PLAN_E17 rev 2 (2026-10-26)", "phase0": {}, "launches": {}, "arms": {}, "order": [], "skipped": {},
       "hook_sha256": E17_HOOK_SHA}
E14_RESULTS["e17"] = E17
E17_OUT = E14_OUT / "e17"
E17_OUT.mkdir(parents=True, exist_ok=True)
E17_HOOK_DIR = Path(os.environ.get("E17_HOOK_DIR", "/tmp/e17_hook"))
E17_HOOK_DIR.mkdir(parents=True, exist_ok=True)
(E17_HOOK_DIR / "sitecustomize.py").write_bytes(E17_HOOK_SRC.encode("utf-8"))      # bytes: no newline translation
assert hashlib.sha256((E17_HOOK_DIR / "sitecustomize.py").read_bytes()).hexdigest() == E17_HOOK_SHA
_SPEC = ("--speculative-algorithm", "--speculative-draft-model-path", "--speculative-draft-model-quantization",
         "--speculative-num-draft-tokens", "--speculative-num-steps", "--speculative-eagle-topk")
E17_TGT_DROP = _SPEC + ("--min-free-slots-delay",)
E17_TOL_FLOOR = 5e-3          # audit 4 (three GPT audits 2026-10-26): a FIXED reuse tolerance (never scaled by the A/A)
E17_AA_MAX = 1e-3             # a reference whose own A/A KL exceeds this is unstable: verdict INCONCLUSIVE, never clean
E17_P = {"W44": [44] * 128, "W42": [42] * 128}
E17_P["P255"] = [42] * 128 + [43] * 127          # upstream Class 1: prefill checkpoint 192, first decode crosses 256
E17_P["P511"] = [42] * 128 + [43] * 383          # interval-256 variant: checkpoint 448, first decode crosses 512
for _n in (246, 249, 252):                        # T1c (interval 64): checkpoint 192, boundary 256 is 4-10 tokens ahead
    E17_P[f"P{_n}"] = [42] * 128 + [43] * (_n - 128)
E17_CTX = None
E17_ARM = {"name": None}


def e17_save():
    e14_save()
    try:
        write_json_atomic(E17_OUT / "e17_report.json", E17)
    except Exception as _e:
        print("E17 save failed:", repr(_e)[:200], flush=True)


def e17_edit_flags(flags, drop=(), set_=None):
    """drop: flag names removed with their value(s); set_: {flag: value | [values] | None (bare flag)}."""
    out, i, set_ = [], 0, dict(set_ or {})
    names = set(drop) | set(set_)
    while i < len(flags):
        f = flags[i]
        if f in names:
            i += 1
            while i < len(flags) and not str(flags[i]).startswith("--"):
                i += 1
            continue
        out.append(f)
        i += 1
    for k, v in set_.items():
        out += [k] + ([str(x) for x in v] if isinstance(v, (list, tuple)) else ([str(v)] if v is not None else []))
    return out


def e17_left():
    return HARD_STOP - time.perf_counter()


def e17_ctx(cap):
    c = E14M.Ctx(SGL_URL, lambda: E11_SERVER.log_text() if E11_SERVER is not None else "", E17_OUT,
                 model_id="e17", deadline=time.perf_counter() + min(cap, max(30.0, e17_left() - 5 * 60)), log=print,
                 encode=None, normalize=lambda x: x)
    c.scale = E14_TS
    c.headers = {"Content-Type": "application/json"}      # no probe header: the real SGLang ignores it, the fake must not
    return c


def e17_launch(name, set_=None, drop=(), env=None, hook=True, verbose=True, timeout=None):
    """Kill the current server, launch the production command + this arm's changes (hook on: PYTHONPATH = the hook dir,
    E17_TRACE_DIR = a fresh per-launch dir), wait, record caps and the hook self-test."""
    global E11_SERVER, E17_CTX
    rec = {"arm": name, "set": {k: v for k, v in (set_ or {}).items()}, "drop": list(drop), "env": dict(env or {}),
           "hook": hook}
    if not E14M.launch_allowed(time.perf_counter(), HARD_STOP, relaunch_cost=240.0, floor=8 * 60, margin=4 * 60):
        rec["skipped"] = "NOT_TESTED_BUDGET"
        E17["launches"][name] = rec
        e17_save()
        print(f"E17 launch {name}: skipped (budget)", flush=True)
        return False, rec
    E14_STATE.enter("E17_LAUNCH:" + name, deadline=HARD_STOP)
    t0 = time.perf_counter()
    try:
        e11_kill_server(wait=True)
    except Exception:
        pass
    E11_SERVER = None
    rec["gpu_free"] = _e11_gpu_free()
    flags = e17_edit_flags(e11_flags(E14_VENV["cfg"]), drop=drop, set_=set_)
    envx = dict(E14_VENV["env"])
    envx.update(SGLANG_OPT_MAMBA_SKIP_DECODE_LOCK="0", SGLANG_TEST_MAMBA_LAZY_ALLOC_FAIL="0")
    tdir = E17_OUT / "trace" / name
    if hook:
        shutil.rmtree(tdir, ignore_errors=True)
        tdir.mkdir(parents=True, exist_ok=True)
        envx.update(PYTHONPATH=str(E17_HOOK_DIR), E17_TRACE_DIR=str(tdir), E17_TRACE_VERBOSE="1" if verbose else "0")
    else:
        envx.pop("PYTHONPATH", None)
        envx.pop("E17_TRACE_DIR", None)
    envx.update(env or {})
    srv = E11Server(E14_VENV["launch"] + flags, envx, "e17_" + name)
    srv.start()
    try:
        ok, why = srv.wait_ready(min(timeout or SGL_RELAUNCH_TIMEOUT_SEC, max(60.0, e17_left() - 6 * 60)))
    except BaseException:
        try:
            srv.kill()
        except Exception:
            pass
        raise
    rec.update(ok=ok, why=why, sec=round(time.perf_counter() - t0, 1), flags=flags, trace_dir=str(tdir) if hook else None)
    if ok:
        E11_SERVER = srv
        rec["caps"] = e11_caps(srv.log_text())
        if hook:
            rec["hook_selftest"] = e17_hook_selftest(tdir)
    else:
        rec["log_tail"] = srv.log_text()[-3000:]
        srv.kill()
    E17["launches"][name] = rec
    E17_ARM["name"] = name if ok else None
    e17_save()
    print(f"E17 launch {name}: {why} ({rec['sec']} s) hook={rec.get('hook_selftest')}", flush=True)
    return ok, rec


def e17_hook_selftest(tdir, wait=60.0):
    """The hook is live when EVERY scheduler rank reports the three scheduler targets patched (audit 4: names merged
    over pids could pass a partial install). TP4 needs 4 fully patched processes (the CPU fake has one)."""
    need = {"ScheduleBatch", "SchedulerBatchResultProcessor", "MambaComponent"}
    want_ranks = 1 if E11_HARNESS else 4
    end = time.perf_counter() + wait * max(E14_TS, 0.05)
    seen, full = set(), 0
    while True:
        full, seen = 0, set()
        for p in Path(tdir).glob("counters_*.json"):
            try:
                names = {x.split("@")[0] for x in json.loads(p.read_text()).get("patched") or []}
            except Exception:
                continue
            seen |= names
            full += 1 if need <= names else 0
        if full >= want_ranks or time.perf_counter() > end:
            break
        time.sleep(1.0)
    fails = []
    for p in Path(tdir).glob("trace_*.jsonl"):
        try:
            fails += [json.loads(l) for l in p.read_text().splitlines() if '"patch_failed"' in l]
        except Exception:
            pass
    return {"ok": full >= want_ranks and not fails, "ranks_fully_patched": full, "ranks_wanted": want_ranks,
            "patched": sorted(seen), "patch_failed": fails[:4]}


def e17_trace(name, settle=3.0):
    """The hook counters of one launch. With TP4 every rank runs the scheduler: the primary process = the one with the
    most decisions (counters are per process, never summed)."""
    tdir = E17_OUT / "trace" / name
    time.sleep(settle * max(E14_TS, 0.05) + 0.5)
    procs = {}
    for p in tdir.glob("counters_*.json"):
        try:
            d = json.loads(p.read_text())
            procs[str(d.get("pid"))] = d
        except Exception:
            continue
    key = lambda d: sum((d.get("counters") or {}).get(k, 0) for k in ("optional_alloc_attempts", "donations_unfinished",
                                                                      "donations_finished", "spec_window_reachable"))
    prim = max(procs.values(), key=key) if procs else None
    out = {"processes": len(procs), "primary_pid": (prim or {}).get("pid"), "counters": dict((prim or {}).get("counters") or {}),
           "alloc": (prim or {}).get("alloc"), "per_pid": {k: (v.get("counters") or {}) for k, v in procs.items() if key(v)}}
    evs = []
    if prim is not None:
        tp = tdir / f"trace_{prim['pid']}.jsonl"
        if tp.exists():
            evs = [json.loads(l) for l in tp.read_text(encoding="utf-8").splitlines() if l.strip()]
    out["events_by_kind"] = {}
    for e in evs:
        out["events_by_kind"][e.get("ev")] = out["events_by_kind"].get(e.get("ev"), 0) + 1
    out["stale_events"] = [e for e in evs if e.get("ev") == "donation" and e.get("stale_candidate")][:20]
    out["fail_events"] = [e for e in evs if e.get("ev") == "alloc_failed"][:20]
    ks = []
    for p in tdir.glob("trace_*.jsonl"):
        ks += [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if '"kv_scales"' in l]
    if ks:
        out["kv_scales"] = ks[0]
    return out


def e17_rid_evidence(tr, name):
    """Per-request evidence of one launch from the primary pid's full trace: rids whose optional slot failed, and rids
    with a CONFIRMED stale donation (decode path: at donation; spec path: a later spec_resolve with crossed=True), with
    the donated length(s)."""
    out = {"failed": set(), "confirmed": {}}
    pid = (tr or {}).get("primary_pid")
    tp = E17_OUT / "trace" / name / f"trace_{pid}.jsonl"
    if pid is None or not tp.exists():
        return out
    pend = {}
    for l in tp.read_text(encoding="utf-8").splitlines():
        try:
            e = json.loads(l)
        except Exception:
            continue
        rid = e.get("rid")
        if e.get("ev") == "alloc_failed" and rid:
            out["failed"].add(rid)
        elif e.get("ev") == "donation" and e.get("stale_candidate") and rid:
            ln = e.get("returned") if isinstance(e.get("returned"), int) else e.get("cache_len")
            if ((e.get("marked") or {}).get("path")) == "decode":
                out["confirmed"].setdefault(rid, []).append(ln)
            else:
                pend.setdefault(rid, []).append(ln)
        elif e.get("ev") == "spec_resolve" and rid and e.get("confirmed"):
            out["confirmed"].setdefault(rid, []).extend(pend.pop(rid, []))
    return out


def e17_idle(ctx, wait=30.0):
    """Idle only when /metrics really reports running and queued = 0 (audit 4: missing fields are not idle)."""
    end = time.monotonic() + max(0.5, wait * max(E14_TS, 0.05))
    while True:
        try:
            p = E14M.prom(ctx, timeout=5)
            if "sglang:num_running_reqs" in p and "sglang:num_queue_reqs" in p and not p["sglang:num_running_reqs"] \
                    and not p["sglang:num_queue_reqs"]:
                return True
        except Exception:
            pass
        if time.monotonic() >= end:
            return False
        time.sleep(0.25)


# ---------------------------------------------------------------------------------------------- probes
def e17_post(ctx, body, timeout=600):
    r = E14M.post_json(ctx, "/generate", body, timeout=timeout)
    return r[0] if isinstance(r, list) else r


def e17_gen(ctx, ids, max_new, *, score=False, ignore_eos=True, temperature=0.0, rid=None):
    sp = {"temperature": temperature, "max_new_tokens": int(max_new), "ignore_eos": ignore_eos}
    if temperature > 0:
        sp.update(top_p=0.95, top_k=20)
    body = {"rid": rid or f"e17-{_e17_rnd.randrange(10 ** 12)}", "input_ids": list(ids), "sampling_params": sp}
    if score:
        body.update(return_logprob=True, logprob_start_len=-1, token_ids_logprob=E17_VOCAB_IDS)
    r = e17_post(ctx, body)
    meta = r.get("meta_info") or {}
    out = {"cached": meta.get("cached_tokens"), "n_out": len(r.get("output_ids") or []), "out": list(r.get("output_ids") or []),
           "finish": meta.get("finish_reason")}
    if score:
        out["lp"] = e17_full_lp(meta)
    return out


def e17_full_lp(meta):
    rows = meta.get("output_token_ids_logprobs") or []
    assert len(rows) >= 1, "no output_token_ids_logprobs"
    pairs = rows[0]
    assert len(pairs) == E17_V, (len(pairs), E17_V)
    ids = [int(x[1]) for x in pairs]
    assert ids == E17_VOCAB_IDS, "vocabulary order differs"
    lp = np.asarray([float(x[0]) for x in pairs], dtype=np.float64)
    assert np.isfinite(lp).all(), "non-finite logprobs"
    s = float(np.exp(lp).sum())
    lp = lp - np.logaddexp.reduce(lp)
    return {"lp": lp, "sum_raw": round(s, 6)}


def e17_kl(a, b):
    """KL(a || b) over the full vocabulary (a = the reference), plus max |dlogp| and top-1 change."""
    pa, la, lb = np.exp(a["lp"]), a["lp"], b["lp"]
    return {"kl": float(np.sum(pa * (la - lb))), "max_abs_dlogp": float(np.max(np.abs(la - lb))),
            "top1_changed": int(np.argmax(la)) != int(np.argmax(lb))}


def e17_flush(ctx):
    """flush_cache on a server that /metrics shows idle; valid = really flushed AND idle (audit 4: a cold control on a
    busy or unflushed server is no control)."""
    f = E14M.flush_cache(ctx, wait=30.0)
    if not f.get("ok"):
        time.sleep(2.0)
        f = E14M.flush_cache(ctx, wait=30.0)
    f["idle_metrics"] = e17_idle(ctx, wait=10.0)
    f["valid"] = bool(f.get("ok") and f.get("idle_metrics"))
    return f


def e17_warm(ctx, prefixes):
    return [e17_gen(ctx, p, 1)["cached"] for p in prefixes]


def e17_t1(ctx, pname, prefixes, seed_new, aa=3, cycles=3):
    """Upstream Class 1 (test_mamba_cache_kl.py): reference = flush, warm the prefixes, score P (aa times = the A/A);
    cycle = flush, warm, seed (P + seed_new greedy tokens), score P again (the reuse)."""
    P = E17_P[pname]
    tag = f"{pname}-{_e17_rnd.randrange(10 ** 9)}"
    out = {"prompt": pname, "len": len(P), "seed_new": seed_new, "ref_cached": [], "reuse_cached": [], "seed_cached": [],
           "flush_ok": [], "aa": [], "reuse": [], "cycles": []}
    out["safe_depth"] = max(len(p) for p in prefixes if P[:len(p)] == p) if prefixes else 0
    refs, ref_flush = [], []
    for _ in range(aa):
        f = e17_flush(ctx)
        ref_flush.append(f["valid"])
        out["flush_ok"].append(f.get("ok"))
        e17_warm(ctx, prefixes)
        r = e17_gen(ctx, P, 1, score=True)
        refs.append(r["lp"])
        out["ref_cached"].append(r["cached"])
        out["sum_raw"] = r["lp"]["sum_raw"]
    out["aa"] = [e17_kl(refs[0], x) for x in refs[1:]]
    out["aa_kl_max"] = max([x["kl"] for x in out["aa"]] or [0.0])
    out["tol"] = E17_TOL_FLOOR
    # audit 4: the reference must be clean (every flush valid, every reference hit exactly the safe prefix) and stable
    out["reference_valid"] = bool(all(ref_flush) and all(c == out["safe_depth"] for c in out["ref_cached"])
                                  and np.isfinite(out["aa_kl_max"]) and out["aa_kl_max"] <= E17_AA_MAX)
    for c in range(cycles):
        f = e17_flush(ctx)
        out["flush_ok"].append(f.get("ok"))
        e17_warm(ctx, prefixes)
        seed_rid, probe_rid = f"e17-seed-{tag}-{c}", f"e17-probe-{tag}-{c}"
        s = e17_gen(ctx, P, seed_new, rid=seed_rid)
        out["seed_cached"].append(s["cached"])
        r = e17_gen(ctx, P, 1, score=True, rid=probe_rid)
        kl = e17_kl(refs[0], r["lp"])
        out["reuse_cached"].append(r["cached"])
        out["reuse"].append(kl)
        out["cycles"].append({"cycle": c, "seed_rid": seed_rid, "probe_rid": probe_rid, "flush_valid": f["valid"],
                              "seed_cached": s["cached"], "probe_cached": r["cached"], "probe_kl": kl["kl"],
                              "top1_changed": kl["top1_changed"]})
    out["reuse_kl_max"] = max([x["kl"] for x in out["reuse"]] or [0.0])
    out["stale_depth_hits"] = sum(1 for c in out["reuse_cached"] if isinstance(c, int) and c > out["safe_depth"])
    out["_ref_lp"] = refs[0]
    return out


def e17_cycle_chain(t, cyc, ev):
    """audit 4: the causal chain of ONE seed/probe cycle: this seed's optional slot failed, this seed made a confirmed
    stale donation of length L, and this probe (after the cycle's own flush) hit exactly L > the safe prefix (the only
    node of that depth in a flushed cache) = the restore link."""
    lens = [x for x in (ev.get("confirmed") or {}).get(cyc["seed_rid"], []) if isinstance(x, int)]
    link = isinstance(cyc["probe_cached"], int) and cyc["probe_cached"] > t["safe_depth"] and cyc["probe_cached"] in lens
    return {"failed": cyc["seed_rid"] in ev.get("failed", set()), "stale_confirmed": bool(lens), "stale_lens": lens,
            "restore_link": link, "hot": cyc["probe_kl"] > t["tol"]}


def e17_verdict(t1s, ev, hook=True):
    """Per cycle, fail-closed (audit 4): BUG_EXPOSED only when ONE valid cycle holds the whole chain (failure -> confirmed
    stale donation -> restore link -> KL above the fixed tolerance). Evidence from different cycles never adds up."""
    if not t1s:
        return "NO_DATA"
    if not all(t["reference_valid"] for t in t1s):
        return "INCONCLUSIVE_UNSTABLE_OR_INVALID_REFERENCE"
    valid = [(t, c) for t in t1s for c in t["cycles"] if c["flush_valid"]]
    if not valid:
        return "INCONCLUSIVE_INVALID_COLD_CONTROL"
    if not hook:
        return "KL_ABOVE_TOL" if any(c["probe_kl"] > t["tol"] for t, c in valid) else "KL_WITHIN_TOL"
    ch = []
    for t, c in valid:
        c["chain"] = e17_cycle_chain(t, c, ev)
        ch.append(c["chain"])
    if any(x["failed"] and x["stale_confirmed"] and x["restore_link"] and x["hot"] for x in ch):
        return "BUG_EXPOSED"
    if any(x["failed"] and x["stale_confirmed"] and x["restore_link"] for x in ch):
        return "STALE_RESTORE_EXPOSED_NO_OUTPUT_FLIP"
    if any(x["hot"] for x in ch):
        return "KL_ABOVE_TOL_CHAIN_INCOMPLETE"
    if any(x["failed"] for x in ch) and not any(x["stale_confirmed"] for x in ch):
        return "TRIGGER_REACHED_NO_STALE"
    return "INCONCLUSIVE"


def e17_strip(t):
    return {k: v for k, v in t.items() if not k.startswith("_")}


def e17_safety(ctx, ref_t1):
    """Patch-safety cases on the 4-slot server (audits 1-3): normal EOS, abort mid-decode, max_new reached, 5 x reuse,
    cold after flush. After each: flush, the allocator's available slots vs the baseline (a leak shows as fewer), then
    warm + score P (a hit deeper than the safe prefix with KL above the tolerance = a stale restore)."""
    P, pref, ref = E17_P["P255"], [E17_P["W44"], E17_P["W42"]], ref_t1["_ref_lp"]
    tdir = Path(E17["launches"][E17_ARM["name"]].get("trace_dir") or E17_OUT)

    def avail():
        time.sleep(3.0 * max(E14_TS, 0.05) + 0.3)
        vals = []
        for p in tdir.glob("counters_*.json"):
            try:
                a = json.loads(p.read_text()).get("alloc") or {}
                if a.get("available") is not None:
                    vals.append(a["available"])
            except Exception:
                pass
        return max(vals) if vals else None
    e17_flush(ctx)
    base = avail()
    rows = []

    def after(case, extra=None):
        # the probe first (the case's own donation is still in the cache), then flush + the slot count
        e17_warm(ctx, pref)
        r = e17_gen(ctx, P, 1, score=True)
        kl = e17_kl(ref, r["lp"])
        f = e17_flush(ctx)
        a = avail()
        rows.append(dict({"case": case, "flush_ok": f.get("ok"), "flush_valid": f.get("valid"), "available": a, "available_base": base,
                          "leak": (a is not None and base is not None and a < base), "probe_cached": r["cached"],
                          "probe_kl": kl["kl"], "stale": bool(r["cached"] and r["cached"] > 128 and kl["kl"] > ref_t1["tol"])},
                         **(extra or {})))
    r = e17_gen(ctx, P, 64, ignore_eos=False)
    after("eos", {"finish": r["finish"]})
    rid = f"e17-abort-{_e17_rnd.randrange(10 ** 9)}"
    th = threading.Thread(target=lambda: e17_gen(ctx, P, 400, rid=rid), daemon=True)
    th.start()
    time.sleep(0.8)
    E14M.abort(ctx, [rid])
    th.join(60)
    after("abort")
    r = e17_gen(ctx, P, 48)
    after("max_new", {"n_out": r["n_out"]})
    for _ in range(5):
        e17_gen(ctx, P, 8)
    after("reuse5")
    e17_flush(ctx)
    r = e17_gen(ctx, P, 1, score=True)
    after("cold", {"cold_cached": r["cached"], "cold_kl": e17_kl(ref, r["lp"])["kl"]})
    return {"rows": rows, "leaks": sum(1 for x in rows if x["leak"]), "stale": sum(1 for x in rows if x["stale"]),
            "cold_cached_zero": rows[-1].get("cold_cached") == 0}


def e17_phase(name, fn, cap):
    """One measured block with its own deadline; recorded even on error; skipped when the budget is gone."""
    if e17_left() < 6 * 60:
        E17["skipped"][name] = "NOT_TESTED_BUDGET"
        e17_save()
        return None
    E14_STATE.enter("E17:" + name, deadline=HARD_STOP)
    t0 = time.perf_counter()
    try:
        res = fn(e17_ctx(cap))
    except BaseException as _e:
        res = {"error": f"{type(_e).__name__}: {_e}"[:500], "traceback": traceback.format_exc()[-1500:]}
    if isinstance(res, dict):
        res["phase_sec"] = round(time.perf_counter() - t0, 1)
        if E11_SERVER is not None:
            res["server_alive_after"] = E11_SERVER.alive()
    E17["arms"][name] = res
    E17["order"].append(name)
    e17_save()
    print(f"E17 {name}: {json.dumps(res, default=str)[:1200]}", flush=True)
    return res


def e17_small(interval, slots):
    return {"--max-mamba-cache-size": slots, "--max-running-requests": 1, "--mamba-track-interval": interval,
            "--chunked-prefill-size": 2048, "--cuda-graph-bs-decode": [1]}


def e17_small_launch(name, interval, spec, env=None, hook=True):
    """The small-slot server (upstream: 4 Mamba slots, 1 running). A failed launch steps up 4 -> 6 -> 8 slots."""
    for slots in (4, 6, 8):
        s = e17_small(interval, slots)
        ok, rec = e17_launch(f"{name}_m{slots}" if slots != 4 else name, set_=s,
                             drop=(("--cuda-graph-max-bs-decode",) + (() if spec else E17_TGT_DROP)), env=env, hook=hook)
        if ok or rec.get("skipped"):
            return ok, rec, slots
    return False, rec, slots


def e17_t1_arm(name, interval, prompts, seed_new, spec=False, env=None, hook=True, safety=False):
    ok, rec, slots = e17_small_launch(name, interval, spec, env=env, hook=hook)
    lname = rec.get("arm", name)
    if not ok:
        E17["arms"][name] = {"launch_ok": False, "why": rec.get("why") or rec.get("skipped"), "slots": slots}
        E17["order"].append(name)
        e17_save()
        return E17["arms"][name]
    if hook and not (rec.get("hook_selftest") or {}).get("ok"):
        E17["arms"][name] = {"launch_ok": True, "verdict": "HOOK_MISSING", "hook_selftest": rec.get("hook_selftest")}
        E17["order"].append(name)
        e17_save()
        return E17["arms"][name]

    def run(ctx):
        t1s = [e17_t1(ctx, p, [E17_P["W44"], E17_P["W42"]], seed_new) for p in prompts]
        res = {"launch": lname, "slots": slots, "interval": interval, "spec": spec, "env": dict(env or {}), "hook": hook,
               "t1": [e17_strip(t) for t in t1s]}
        if safety:
            res["safety"] = e17_safety(ctx, t1s[0])
        tr = e17_trace(lname) if hook else None
        res["trace"] = tr
        ev = e17_rid_evidence(tr, lname) if hook else {"failed": set(), "confirmed": {}}
        res["verdict"] = e17_verdict(t1s, ev, hook=hook)
        res["t1"] = [e17_strip(t) for t in t1s]          # again: e17_verdict added the per-cycle chain records
        res["rid_evidence"] = {"failed": sorted(ev["failed"])[:40], "confirmed": dict(list(ev["confirmed"].items())[:40])}
        if spec:
            res["accept"] = [l for l in (E11_SERVER.log_text() if E11_SERVER else "").splitlines() if "accept len" in l][-3:]
        return res
    return e17_phase(name, run, 14 * 60)


# ---------------------------------------------------------------------------------------------- Phase 0
def e17_pkg_dir():
    if E11_HARNESS:
        return Path(os.environ["E14_FAKE_SGLANG_PKG"])
    r = subprocess.run([E14_VENV["launch"][0], "-c", "import sglang, os; print(os.path.dirname(sglang.__file__))"],
                       capture_output=True, text=True, timeout=300, env=E14_VENV["env"])
    return Path(r.stdout.strip().splitlines()[-1])


def e17_phase0():
    p0 = E17["phase0"]
    try:
        pkg = e17_pkg_dir()
        p0["pkg_dir"] = str(pkg)
        sb = pkg / "srt" / "managers" / "schedule_batch.py"
        p0["schedule_batch_sha256"] = hashlib.sha256(sb.read_bytes()).hexdigest()
        p0["patch_43567_check"] = E14P.check(str(pkg), ("43567",))
        p0["patch_43567_state"] = p0["patch_43567_check"]["patches"]["43567"]["state"]
    except Exception as e:
        p0["pkg_error"] = repr(e)[:300]
    try:
        dc = _load_json(Path(SGL_DRAFT_PATH) / "config.json")
        tl = (dc.get("dflash_config") or {}).get("target_layer_ids")
        p0["dflash_target_layer_ids"], p0["dflash_num_target_layers"] = tl, dc.get("num_target_layers")
        p0["dflash_taps_ok"] = tl == [5, 19, 33, 47, 61] and dc.get("num_target_layers") == 64
        p0["dflash_block_size"] = (dc.get("dflash_config") or {}).get("block_size")
    except Exception as e:
        p0["dflash_taps_ok"], p0["dflash_error"] = False, repr(e)[:300]
    try:
        mc = _load_json(Path(SGL_MODEL_PATH) / "config.json")
        v = mc.get("vocab_size") or (mc.get("text_config") or {}).get("vocab_size")
        p0["vocab_size_config"] = v
        idx = Path(SGL_MODEL_PATH) / "model.safetensors.index.json"
        if idx.exists():
            wm = (_load_json(idx).get("weight_map") or {})
            p0["checkpoint_kv_scale_keys"] = sorted(k for k in wm if k.endswith(("k_scale", "v_scale")))[:8]
            p0["checkpoint_kv_scale_n"] = sum(1 for k in wm if k.endswith(("k_scale", "v_scale")))
    except Exception as e:
        p0["vocab_error"] = repr(e)[:300]
    lt = E11_SERVER.log_text() if E11_SERVER is not None else ""
    p0["gdn_backend_line"] = "Using hybrid linear attention backend for hybrid GDN models" in lt
    p0["kda_backend_line"] = "KDAAttnBackend" in lt
    p0["kv_scale_log_lines"] = [l[-200:] for l in lt.splitlines() if "scaling factor" in l.lower() or "k_scale" in l][:6]
    p0["production_engine_ready"] = bool(ENGINE_READY)
    p0["production_flags"] = [str(x) for x in (E11_SGL.get("launch_flags") or [])]
    e17_save()
    print("E17 phase 0:", json.dumps({k: v for k, v in p0.items() if k not in ("patch_43567_check", "production_flags")},
                                      default=str)[:1500], flush=True)
    return p0


# ---------------------------------------------------------------------------------------------- T3b / T3a / L6 / T2
def e17_arc_prompts(n=16):
    ask = "Find the transformation rule of this ARC-AGI-2 task and describe it, then give the test output grids."

    def rows(g):
        return "\n".join(" ".join(str(v) for v in r) for r in g)

    def text(t):
        s = "".join(f"\nExample {i + 1}\nInput:\n{rows(p['input'])}\nOutput:\n{rows(p['output'])}\n" for i, p in enumerate(t["train"]))
        return s + "".join(f"\nTest input {j + 1}:\n{rows(q['input'])}\n" for j, q in enumerate(t["test"]))
    try:
        allt = _load_json(ARC_ROOT / "arc-agi_evaluation_challenges.json")
    except Exception:
        allt = dict(CHALLENGES)
    keys = sorted(allt)
    msgs = [[{"role": "user", "content": ask + "\n" + text(allt[k])}] for k in keys[:n]]
    ids = [tokenizer.encode(tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True, enable_thinking=True,
                                                          reasoning_effort=REASONING_EFFORT), add_special_tokens=False) for m in msgs]
    return ids, keys, text, allt


def e17_long_ids(target_tokens, skip=0):
    """One long user prompt from consecutive evaluation tasks (the T2 shared prompts, the T3a long prefix)."""
    _, keys, text, allt = e17_arc_prompts(1)          # keys = ALL evaluation task ids (sorted), not only the first n
    assert keys, "no ARC evaluation tasks loaded"
    parts, n = [], 0
    for off in range(len(keys)):                       # audit 4: wrap around the pool; never silently short
        parts.append(text(allt[keys[(skip + off) % len(keys)]]))
        n += len(tokenizer.encode(parts[-1], add_special_tokens=False))
        if n >= target_tokens:
            break
    m = [{"role": "user", "content": "Reference ARC tasks:\n" + "\n".join(parts) + "\nDescribe the rule of the last task."}]
    ids = tokenizer.encode(tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True, enable_thinking=True,
                                                         reasoning_effort=REASONING_EFFORT), add_special_tokens=False)
    if len(ids) < target_tokens:
        raise RuntimeError(f"E17 long prompt too short: {len(ids)} < {target_tokens} (skip {skip}, {len(parts)} tasks)")
    return ids


def e17_text_bad(s):
    s = (s or "").strip()
    if not s:
        return True
    if re.search(r"(.)\1{31,}", s):
        return True
    return len(s) >= 200 and (len(set(s)) < 8 or sum(ch.isalnum() for ch in s) / len(s) < 0.10)


def e17_rows_summary(rows):
    ids = [r.get("ids") or [] for r in rows]
    return {"n": len(rows), "errors": sum(1 for r in rows if r.get("error")), "empty": sum(1 for x in ids if not x),
            "degenerate": sum(1 for x in ids if x and e17_text_bad(tokenizer.decode(x, skip_special_tokens=True))),
            "tokens": sum(len(x) for x in ids), "accept": [r.get("accept") for r in rows][:4]}


E17_T3 = {}


def e17_t3b_target(ctx):
    S4, S16 = E17_T3["ids"][:4], E17_T3["ids"]
    a = E14M.parity_pass(ctx, S4, max_new=E17_T3["n"], tag="t3b_tgt_serial_a")
    b = E14M.parity_pass(ctx, S4, max_new=E17_T3["n"], tag="t3b_tgt_serial_b")
    c = E14M.parity_pass(ctx, S16, max_new=E17_T3["n"], concurrent=True, tag="t3b_tgt_conc")
    E17_T3["tgt"] = (a, b, c)
    out = {"serial_aa": E14M.compare_passes(a, b, "target serial A/A"),
           "target_conc_vs_serial": E14M.compare_passes(a, c[:4], "target 16-conc vs serial"),
           "rows": {"serial_a": e17_rows_summary(a), "conc": e17_rows_summary(c)}}
    if e17_left() > 20 * 60:       # T3a fp16 half (radix is off on this server); the arrays stay out of the JSON
        E17_T3["fp16"] = e17_t3a_scores(ctx)
        out["t3a_fp16"] = {k: {"cached": v["cached"], "aa_kl": e17_kl(v["a"], v["b"])["kl"]} for k, v in E17_T3["fp16"].items()}
    return out


def e17_t3b_dflash(ctx):
    S4, S16 = E17_T3["ids"][:4], E17_T3["ids"]
    s = E14M.parity_pass(ctx, S4, max_new=E17_T3["n"], tag="t3b_dfl_serial")
    c = E14M.parity_pass(ctx, S16, max_new=E17_T3["n"], concurrent=True, tag="t3b_dfl_conc")
    out = {"rows": {"serial": e17_rows_summary(s), "conc": e17_rows_summary(c)}}
    if E17_T3.get("tgt"):
        ta, tb, tc = E17_T3["tgt"]
        out.update(dflash_serial_vs_target_serial=E14M.compare_passes(ta, s, "DFlash serial vs target serial"),
                   dflash_conc_vs_target_conc=E14M.compare_passes(tc, c, "DFlash 16-conc vs target 16-conc"),
                   dflash_conc_vs_dflash_serial=E14M.compare_passes(s, c[:4], "DFlash 16-conc vs DFlash serial"))
        out["target_serial_aa"] = E14M.compare_passes(ta, tb, "target serial A/A")
        out["verdict"] = e17_t3b_verdict(out, out["target_serial_aa"])
    out["accept_lines"] = [l[-160:] for l in (E11_SERVER.log_text() if E11_SERVER else "").splitlines() if "accept len" in l][-3:]
    return out


def e17_t3b_verdict(out, out_aa):
    """audit 4 (fail-closed): every comparison complete (expected > 0, every row compared, no error, no missing margin),
    the target A/A too; only then the large-margin flips decide."""
    comps = [out.get(k) or {} for k in ("dflash_serial_vs_target_serial", "dflash_conc_vs_target_conc", "dflash_conc_vs_dflash_serial")]
    comps.append(out_aa or {})
    ok = all((c.get("expected") or 0) > 0 and c.get("compared") == c.get("expected") and c.get("errors", 1) == 0
             and c.get("missing_margins", 1) == 0 for c in comps)
    if not ok:
        return "INCOMPLETE"
    aa = (out_aa or {}).get("large_margin_flips") or 0
    return "DFLASH_HIGH_MARGIN_DIVERGENCE" if max(c.get("large_margin_flips") or 0 for c in comps[:2]) > aa + 1 \
        else "NO_HIGH_MARGIN_DIVERGENCE"


def e17_t3a_scores(ctx):
    ids, out = E17_T3["long"], {}
    for cut in E17_T3["cuts"]:
        if cut > len(ids):
            continue
        a = e17_gen(ctx, ids[:cut], 1, score=True)
        b = e17_gen(ctx, ids[:cut], 1, score=True)
        out[str(cut)] = {"a": a["lp"], "b": b["lp"], "cached": [a["cached"], b["cached"]]}
    return out


def e17_t3a_compare(fp16, fp32):
    rows = {}
    for cut in fp16:
        if cut in fp32:
            rows[cut] = {"aa_fp16": e17_kl(fp16[cut]["a"], fp16[cut]["b"])["kl"], "aa_fp32": e17_kl(fp32[cut]["a"], fp32[cut]["b"])["kl"],
                         "fp16_vs_fp32": e17_kl(fp32[cut]["a"], fp16[cut]["a"])}
    return rows


def e17_l6(ctx):
    """Upstream Class 2 (TestMambaSpecTruncationKL): no_buffer + MTP, a short seed, probe = prompt + output + newline,
    warm vs cold. 0.5.21 has no #43387 guard: a large KL here means the harness can see a stale restore."""
    res = {}
    for case, q, n in (("count", "Count from 1 to 20, separated by commas.", 9), ("story", "Tell a short story about a cat and a fish.", 5)):
        ids = tokenizer.encode(tokenizer.apply_chat_template([{"role": "user", "content": q}], tokenize=False,
                                                             add_generation_prompt=True, enable_thinking=False), add_special_tokens=False)
        e17_flush(ctx)
        s = e17_gen(ctx, ids, n)
        probe = ids + s["out"] + tokenizer.encode("\n", add_special_tokens=False)
        w = e17_gen(ctx, probe, 1, score=True)
        e17_flush(ctx)
        c = e17_gen(ctx, probe, 1, score=True)
        e17_flush(ctx)
        c2 = e17_gen(ctx, probe, 1, score=True)
        res[case] = {"seed_out": s["n_out"], "seed_finish": s["finish"], "warm_cached": w["cached"], "cold_cached": [c["cached"], c2["cached"]],
                     "kl_cold_warm": e17_kl(c["lp"], w["lp"]), "kl_cold_cold": e17_kl(c["lp"], c2["lp"])["kl"]}
    hot = [k for k, v in res.items() if v["warm_cached"] and v["kl_cold_warm"]["kl"] > max(E17_TOL_FLOOR, 10 * v["kl_cold_cold"])]
    res["verdict"] = "STALE_RESTORE_DETECTED" if hot else "NOT_DETECTED"
    return res


def e17_t2(ctx, tag, minutes):
    """T2: 16 streams, 4 shared task prompts x 4 streams, production sampling; each turn appends its output and a
    feedback block (every turn is a prefix hit; > 45k restarts the chain from the shared prompt = the Poetiq pattern).
    At the end: per shared prompt a warm probe, then flush and two cold probes (full vocabulary)."""
    bases, fb = E17_T2["bases"], E17_T2["fb"]
    stop_at = time.perf_counter() + minutes * 60 * max(E14_TS, 0.02)
    stats = {"requests": 0, "prefix_hits": 0, "errors": 0, "tokens_out": 0, "restarts": 0, "cached_sum": 0, "prompt_sum": 0,
             "error_samples": []}
    lock = threading.Lock()
    stop_ev, active = threading.Event(), set()

    def stream(si):
        ids = list(bases[si % 4])
        n = 0
        while time.perf_counter() < stop_at - 120 * max(E14_TS, 0.02) and not stop_ev.is_set():
            rid = f"e17-t2-{tag}-{si}-{n}"
            n += 1
            with lock:
                active.add(rid)
            try:
                r = e17_gen(ctx, ids, E17_T2["max_new"], temperature=1.0, ignore_eos=False, rid=rid)
            except Exception as e:
                with lock:
                    stats["errors"] += 1
                    if len(stats["error_samples"]) < 6:
                        stats["error_samples"].append(repr(e)[:240])
                time.sleep(1.0)
                continue
            finally:
                with lock:
                    active.discard(rid)
            with lock:
                stats["requests"] += 1
                stats["prefix_hits"] += 1 if (r["cached"] or 0) > 0 else 0
                stats["tokens_out"] += r["n_out"]
                stats["cached_sum"] += r["cached"] or 0
                stats["prompt_sum"] += len(ids)
            ids = ids + r["out"] + fb
            if len(ids) > E17_T2["max_ctx"]:
                ids = list(bases[si % 4])
                with lock:
                    stats["restarts"] += 1
    ths = [threading.Thread(target=stream, args=(i,), daemon=True) for i in range(16)]
    for t in ths:
        t.start()
    E14M.join_all(ths, max(1.0, stop_at - time.perf_counter() + 60))
    # audit 4: the cleanup gate before any probe: stop new turns, abort what still runs, every thread joined, idle
    stop_ev.set()
    with lock:
        live = list(active)
    if live:
        E14M.abort(ctx, live, budget=8)
    alive = E14M.join_all(ths, 40)
    idle = e17_idle(ctx, wait=60)
    base_out = {"stats": stats, "alive_threads": alive, "server_idle": idle, "aborted": len(live),
                "base_lens": [len(b) for b in bases]}
    if alive or not idle or stats["errors"]:
        base_out["verdict"] = "INCOMPLETE"
        base_out["why"] = "cleanup gate: alive threads / server not idle / stream errors"
        base_out["trace"] = e17_trace(tag)
        return base_out
    probes = []
    for bi, b in enumerate(bases):
        w = e17_gen(ctx, b, 1, score=True)
        f1 = e17_flush(ctx)
        c1 = e17_gen(ctx, b, 1, score=True)
        f2 = e17_flush(ctx)
        c2 = e17_gen(ctx, b, 1, score=True)
        probes.append({"base": bi, "len": len(b), "warm_cached": w["cached"], "cold_cached": [c1["cached"], c2["cached"]],
                       "cold_valid": bool(f1["valid"] and f2["valid"] and c1["cached"] == 0 and c2["cached"] == 0),
                       "kl_cold_warm": e17_kl(c1["lp"], w["lp"]), "kl_cold_cold": e17_kl(c1["lp"], c2["lp"])["kl"]})
    tr = e17_trace(tag)
    k = tr.get("counters") or {}
    att, fail = k.get("optional_alloc_attempts", 0), k.get("optional_alloc_failed", 0)
    stale = k.get("stale_donation_confirmed", 0)
    if not all(p["cold_valid"] and p["kl_cold_cold"] <= E17_AA_MAX for p in probes):
        verdict = "INCOMPLETE_COLD_CONTROL"
    else:
        # T2 has no per-request provenance: a warm divergence is reported next to the counters, never as a confirmed chain
        div = any(p["warm_cached"] and p["kl_cold_warm"]["kl"] > E17_TOL_FLOOR for p in probes)
        verdict = ("NO_FAILURE_OBSERVED" if not fail else "FAILURE_OBSERVED_NO_STALE_RESTORE" if not stale else
                   "STALE_DONATIONS_AND_WARM_DIVERGENCE" if div else "STALE_RESTORE_EXPOSED_NO_OUTPUT_FLIP")
    return dict(base_out, probes=probes, trace=tr, verdict=verdict,
                fail_per_1000_boundaries=round(1000 * fail / att, 2) if att else None,
                stale_per_1000_prefix_hits=round(1000 * stale / stats["prefix_hits"], 2) if stats["prefix_hits"] else None)


def e17_revert_verified(pkg):
    """audit 4: kill the server, revert, then require revert ok + state not_applied + the Phase-0 sha of
    schedule_batch.py; otherwise raise (no "unpatched" arm may run on a patched build)."""
    try:
        e11_kill_server(wait=True)
    except Exception:
        pass
    res = E14P.revert(str(pkg))
    state = E14P.check(str(pkg), ("43567",))["patches"]["43567"]["state"]
    sha_now = hashlib.sha256((Path(pkg) / "srt" / "managers" / "schedule_batch.py").read_bytes()).hexdigest()
    rec = {"revert": res, "state": state, "sha_ok": sha_now == E17["phase0"].get("schedule_batch_sha256")}
    E17.setdefault("reverts", []).append(rec)
    if not res.get("ok") or state != "not_applied" or not rec["sha_ok"]:
        E17["patch_revert_failed"] = rec
        e17_save()
        raise RuntimeError(f"PATCH_REVERT_FAILED: ok={res.get('ok')} state={state} sha_ok={rec['sha_ok']}")
    return rec


# ---------------------------------------------------------------------------------------------- run
try:
    E17_ON = bool(P.get("e17")) and not RERUN and bool(E14_VENV.get("launch"))
    if E17_ON:
        e17_phase0()
        E17_V = int(os.environ.get("E17_VOCAB") or E17["phase0"].get("vocab_size_config") or 248320)
        E17_VOCAB_IDS = list(range(E17_V))
        E17["vocab_used"] = E17_V
        E17["phase0"]["gate"] = {"dflash_taps_ok": E17["phase0"].get("dflash_taps_ok"),
                                 "unpatched_at_start": E17["phase0"].get("patch_43567_state") == "not_applied"}
        if not E17["phase0"].get("dflash_taps_ok"):
            E17["notes_dflash"] = "DFlash taps not verified: the DFlash arms still run, their results are flagged"
        PKG17 = Path(E17["phase0"]["pkg_dir"])
        if E17["phase0"].get("patch_43567_state") != "not_applied":
            raise RuntimeError(f"E17: SGLang is not the unpatched original at start: {E17['phase0'].get('patch_43567_state')}")
        # user 2026-10-26 (after ~9 GPU-h of engine sweeps with no speed gain): the first E17 session is SHORT (<= 60
        # min): only the decisive #43567 pairs L1 / L7 (ordinary decode, + the safety cases) and L5 / L9 (DFlash W6).
        # The profile E17 keeps the full plan (L2-L4, L6, T3b, T2, T3a) for a follow-up only if the bug shows.
        E17_SHORT = bool(P.get("e17_short"))
        E17["mode"] = "SHORT" if E17_SHORT else "FULL"
        if not E17_SHORT:
            E17_T3["ids"], _, _, _ = e17_arc_prompts(16)
            E17_T3["n"] = int(os.environ.get("E17_T3B_TOKENS", 2048))
            E17_T3["long"] = e17_long_ids(int(os.environ.get("E17_T3A_TOKENS", 50000)))
            E17_T3["cuts"] = [c for c in (8192, 16384, 32768, 49152) if c < len(E17_T3["long"])] or [len(E17_T3["long"]) - 1]
            E17["t3a_long_len"], E17["t3a_cuts"] = len(E17_T3["long"]), list(E17_T3["cuts"])

        # ---- unpatched block
        e17_t1_arm("L1_t1a_128", 128, ["P255"], 32, safety=True)
        if not E17_SHORT:
            e17_t1_arm("L2_t1a_128_hook_off", 128, ["P255"], 32, hook=False)
            e17_t1_arm("L3_t1a_256", 256, ["P255", "P511"], 32)
            e17_t1_arm("L4_t1b_forced", 128, ["P255"], 32, env={"SGLANG_TEST_MAMBA_LAZY_ALLOC_FAIL": "1"})
        e17_t1_arm("L5_t1c_dflash", 64, ["P246", "P249", "P252"], 160, spec=True)
        if E17_SHORT:
            for _k in ("L2_t1a_128_hook_off", "L3_t1a_256", "L4_t1b_forced", "L6_class2_mtp", "L12_t3b_target", "L13_t3b_dflash",
                       "L8_t1a_256_patched", "T2", "L15_t3a_fp32"):
                E17["skipped"][_k] = "SHORT session (user 2026-10-26): follow-up only if the bug shows"
        elif e17_left() > 70 * 60:
            ok6, _ = e17_launch("L6_class2_mtp", set_={
                "--speculative-algorithm": "EAGLE", "--speculative-draft-model-path": str(SGL_MODEL_PATH),
                "--speculative-num-steps": 3, "--speculative-eagle-topk": 1, "--speculative-num-draft-tokens": 4,
                "--mamba-radix-cache-strategy": "no_buffer", "--page-size": 1, "--disable-overlap-schedule": None,
                "--disable-cuda-graph": None, "--max-running-requests": 8, "--max-mamba-cache-size": 128},
                drop=("--cuda-graph-bs-decode", "--cuda-graph-max-bs-decode", "--disable-prefill-cuda-graph",
                      "--min-free-slots-delay"), hook=False)
            if ok6:
                e17_phase("L6_class2_mtp", e17_l6, 8 * 60)
        else:
            E17["skipped"]["L6_class2_mtp"] = "NOT_TESTED_BUDGET"
        if not E17_SHORT:
            okt, _ = e17_launch("L12_t3b_target", set_={"--disable-radix-cache": None}, drop=E17_TGT_DROP, hook=False)
            if okt:
                e17_phase("L12_t3b_target", e17_t3b_target, 22 * 60)
            okd, _ = e17_launch("L13_t3b_dflash", set_={"--disable-radix-cache": None}, hook=False)
            if okd:
                e17_phase("L13_t3b_dflash", e17_t3b_dflash, 14 * 60)

        # ---- #43567 patched block
        E17["patch_apply"] = E14P.apply(str(PKG17), ("43567",))
        E17["patch_check_after_apply"] = E14P.check(str(PKG17), ("43567",))["patches"]["43567"]["state"]
        e17_save()
        print("E17 patch #43567:", E17["patch_apply"].get("ok"), E17["patch_check_after_apply"], flush=True)
        if E17["patch_check_after_apply"] == "applied":
            e17_t1_arm("L7_t1a_128_patched", 128, ["P255"], 32, safety=True)
            if not E17_SHORT:
                e17_t1_arm("L8_t1a_256_patched", 256, ["P255", "P511"], 32)
            e17_t1_arm("L9_t1c_dflash_patched", 64, ["P246", "P249", "P252"], 160, spec=True)
            for _a in ("L7_t1a_128_patched", "L8_t1a_256_patched", "L9_t1c_dflash_patched"):
                _ln = (E17["arms"].get(_a) or {}).get("launch") or _a
                try:
                    _lt = (OUT / f"sglang_server_e17_{_ln}.log").read_text(encoding="utf-8", errors="replace")
                except Exception:
                    _lt = ""
                if "Can not alloc mamba cache" in _lt and isinstance(E17["arms"].get(_a), dict):
                    E17["arms"][_a]["crash_path_43559"] = True       # #43559's assertion: a crash path, not a verdict

        # ---- decision point A
        _v = lambda a: (E17["arms"].get(a) or {}).get("verdict")
        _un = [a for a in ("L1_t1a_128", "L3_t1a_256", "L5_t1c_dflash") if _v(a) == "BUG_EXPOSED"]
        _pa = {"L1_t1a_128": "L7_t1a_128_patched", "L3_t1a_256": "L8_t1a_256_patched", "L5_t1c_dflash": "L9_t1c_dflash_patched"}
        _clean = [a for a in _un if _v(_pa[a]) == "TRIGGER_REACHED_NO_STALE" and
                  ((E17["arms"].get(_pa[a]) or {}).get("trace") or {}).get("counters", {}).get("stale_donation_confirmed", 0) == 0]
        E17["decision_A"] = {"unpatched_bug_arms": _un, "patched_clean": _clean,
                             "verdict": "BUG_EXPOSED_43567" if _clean else ("BUG_SEEN_PATCH_NOT_CLEAN" if _un else "NOT_EXPOSED_OR_INCONCLUSIVE")}
        e17_save()
        print("E17 decision A:", E17["decision_A"], flush=True)

        if E17["decision_A"]["verdict"] == "BUG_EXPOSED_43567" and not E17_SHORT and e17_left() > 40 * 60:
            E17_T2 = {"bases": [e17_long_ids(int(os.environ.get("E17_T2_TOKENS", 30000)), skip=12 * i) for i in range(4)],
                      "fb": tokenizer.encode("\nFeedback: the program failed on example 2; the output grid differs in 3 "
                                             "cells. Revise the rule and try again.\n" * 30, add_special_tokens=False),
                      "max_new": int(os.environ.get("E17_T2_MAX_NEW", 3000)), "max_ctx": 45000}
            E17["t2_base_lens"] = [len(b) for b in E17_T2["bases"]]
            assert len({tuple(b) for b in E17_T2["bases"]}) == 4, "T2 bases are not distinct"
            if E14P.check(str(PKG17), ("43567",))["patches"]["43567"]["state"] == "applied":
                ok11, _ = e17_launch("L11_t2_patched", hook=True, verbose=False)
                if ok11:
                    e17_phase("L11_t2_patched", lambda c: e17_t2(c, "L11_t2_patched", 12), 16 * 60)
            E17["patch_revert"] = e17_revert_verified(PKG17)      # raises: no unpatched arm on a patched build
            ok10, _ = e17_launch("L10_t2_unpatched", hook=True, verbose=False)
            if ok10:
                e17_phase("L10_t2_unpatched", lambda c: e17_t2(c, "L10_t2_unpatched", 12), 16 * 60)
        elif not E17_SHORT:
            E17["skipped"]["T2"] = "decision A: " + E17["decision_A"]["verdict"] if E17["decision_A"]["verdict"] != "BUG_EXPOSED_43567" else "NOT_TESTED_BUDGET"
        if "patch_revert" not in E17:
            E17["patch_revert"] = e17_revert_verified(PKG17)
        E17["patch_state_end"] = E14P.check(str(PKG17), ("43567",))["patches"]["43567"]["state"]

        # ---- T3a fp32 (only with time left)
        if E17_SHORT:
            pass
        elif E17_T3.get("fp16") and e17_left() > 22 * 60:
            ok15, _ = e17_launch("L15_t3a_fp32", set_={"--disable-radix-cache": None, "--mamba-ssm-dtype": "float32"},
                                 drop=E17_TGT_DROP, hook=False)
            if ok15:
                def _t3a(ctx):
                    f32 = e17_t3a_scores(ctx)
                    return {"rows": e17_t3a_compare(E17_T3["fp16"], f32)}
                e17_phase("L15_t3a_fp32", _t3a, 12 * 60)
        else:
            E17["skipped"]["L15_t3a_fp32"] = "NOT_TESTED_BUDGET or no fp16 scores"
        E14_STATE.enter("E17_DONE")
        e17_save()
    else:
        E17["skipped"]["all"] = "rerun" if RERUN else ("no SGLang venv" if not E14_VENV.get("launch") else "E17 profile off")
        print("E17 skipped:", E17["skipped"]["all"], flush=True)
except BaseException:
    E17["error"] = traceback.format_exc()[-3000:]
    print("E17 cell error:", E17["error"][-1500:], flush=True)
    try:
        if "PKG17" in globals() and E14P.check(str(PKG17), ("43567",))["patches"]["43567"]["state"] != "not_applied":
            E17["patch_revert_after_error"] = e17_revert_verified(PKG17)
    except Exception:
        pass
    try:
        e17_save()
    except Exception:
        pass


In [ ]:
# ==================================================================== CELL E17-R — report, bundle
try:
    E14_STATE.enter("REPORT")
    _E = globals().get("E17") or {}
    _p0 = _E.get("phase0") or {}
    L = ["# E17: SGLang silent-accuracy tests (#43567 chain, DFlash parity)", "",
         f"profile {PROFILE}, rerun {RERUN}, engine {ENGINE_INFO.get('backend')}, plan {_E.get('plan')}",
         f"E17 cell error: {(_E.get('error') or 'none')[-800:]}", "",
         "## Phase 0", "",
         f"- schedule_batch.py sha256 {_p0.get('schedule_batch_sha256')}; #43567 at start {_p0.get('patch_43567_state')}; "
         f"after apply {_E.get('patch_check_after_apply')}; at the end {_E.get('patch_state_end')}",
         f"- DFlash taps {_p0.get('dflash_target_layer_ids')} (num_target_layers {_p0.get('dflash_num_target_layers')}, "
         f"block {_p0.get('dflash_block_size')}): ok {_p0.get('dflash_taps_ok')}",
         f"- vocab config {_p0.get('vocab_size_config')}, used {_E.get('vocab_used')}; GDN backend line {_p0.get('gdn_backend_line')}, "
         f"KDA line {_p0.get('kda_backend_line')}",
         f"- checkpoint KV-scale keys {_p0.get('checkpoint_kv_scale_n')} {_p0.get('checkpoint_kv_scale_keys')}; scale log lines "
         f"{_p0.get('kv_scale_log_lines')}", ""]
    _ks = None
    for _a in (_E.get("arms") or {}).values():
        _ks = _ks or (((_a or {}).get("trace") or {}).get("kv_scales"))
    L.append(f"- runtime KV scales (TP0, first {len((_ks or {}).get('rows') or [])} of {(_ks or {}).get('n')} layers): "
             f"{json.dumps(((_ks or {}).get('rows') or [])[:6], default=str)[:900]}")
    L += ["", "## T1 arms (full-vocabulary KL, reference = the same prompt after flush + warm prefixes)", "",
          "| arm | launch | slots / interval | verdict | prompt | ref cached | seed cached | reuse cached | A/A KL max | tol | reuse KL max | "
          "top-1 changed | alloc attempts / failed | stale candidate / confirmed | spec fallback / crossed | safety leaks / stale |",
          "|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|"]
    for _n in _E.get("order") or []:
        _a = (_E.get("arms") or {}).get(_n) or {}
        if "t1" not in _a:
            continue
        _k = ((_a.get("trace") or {}).get("counters") or {})
        _s = _a.get("safety") or {}
        for _t in _a["t1"]:
            L.append(f"| {_n} | {_a.get('launch')} | {_a.get('slots')} / {_a.get('interval')} | {_a.get('verdict')} | {_t['prompt']} | "
                     f"{_t['ref_cached']} | {_t['seed_cached']} | {_t['reuse_cached']} | {_t['aa_kl_max']:.2e} | {_t['tol']:.1e} | "
                     f"{_t['reuse_kl_max']:.3e} | {sum(1 for x in _t['reuse'] if x['top1_changed'])}/{len(_t['reuse'])} | "
                     f"{_k.get('optional_alloc_attempts', 0)} / {_k.get('optional_alloc_failed', 0)} | "
                     f"{_k.get('stale_donation_candidate', 0)} / {_k.get('stale_donation_confirmed', 0)} | "
                     f"{_k.get('spec_inplace_fallback', 0)} / {_k.get('spec_crossed', 0)} | "
                     f"{_s.get('leaks', '')} / {_s.get('stale', '')} |")
    for _n in _E.get("order") or []:
        _a = (_E.get("arms") or {}).get(_n) or {}
        if "t1" not in _a and _n.startswith("L") and _n[1:2].isdigit() and ("verdict" in _a or "launch_ok" in _a or "error" in _a):
            if _n.startswith(("L1_", "L2_", "L3_", "L4_", "L5_", "L7_", "L8_", "L9_")):
                L.append(f"| {_n} | — | {_a.get('slots')} | {_a.get('verdict') or ('LAUNCH_FAILED: ' + str(_a.get('why')) if _a.get('launch_ok') is False else _a.get('error', '')[:200])} | | | | | | | | | | | | |")
    L += ["", f"**Decision A:** {json.dumps(_E.get('decision_A'))}", ""]
    _l6 = (_E.get("arms") or {}).get("L6_class2_mtp")
    L.append(f"**L6 control (no_buffer + MTP, #43387 path):** {json.dumps(_l6, default=str)[:1200] if _l6 else _E.get('skipped', {}).get('L6_class2_mtp', 'not run')}")
    L += ["", "## T2 natural pressure (production flags, 16 streams)", ""]
    for _n in ("L11_t2_patched", "L10_t2_unpatched"):
        _a = (_E.get("arms") or {}).get(_n)
        if _a:
            _k = ((_a.get("trace") or {}).get("counters") or {})
            L.append(f"- {_n}: verdict {_a.get('verdict')}; stats {_a.get('stats')}; failures per 1,000 boundaries "
                     f"{_a.get('fail_per_1000_boundaries')}; stale per 1,000 prefix hits {_a.get('stale_per_1000_prefix_hits')}; "
                     f"counters {_k}; alloc {(_a.get('trace') or {}).get('alloc')}")
            for _pp in _a.get("probes") or []:
                L.append(f"  - base {_pp['base']} ({_pp['len']} tok): warm cached {_pp['warm_cached']}, cold {_pp['cold_cached']}, "
                         f"KL cold||warm {_pp['kl_cold_warm']['kl']:.3e} (top-1 {_pp['kl_cold_warm']['top1_changed']}), cold A/A {_pp['kl_cold_cold']:.3e}")
    if "T2" in (_E.get("skipped") or {}):
        L.append(f"- T2 skipped: {_E['skipped']['T2']}")
    L += ["", "## T3b DFlash vs target-only (radix off, greedy)", ""]
    for _n in ("L12_t3b_target", "L13_t3b_dflash"):
        _a = (_E.get("arms") or {}).get(_n) or {}
        _cmp = {k: {kk: v.get(kk) for kk in ("compared", "expected", "identical", "divergence_p50", "large_margin_flips", "missing_margins")}
                for k, v in _a.items() if isinstance(v, dict) and "large_margin_flips" in v}
        L.append(f"- {_n}: verdict {_a.get('verdict')}; compares {json.dumps(_cmp)}; rows {json.dumps(_a.get('rows'))}; "
                 f"accept {_a.get('accept_lines')}; error {(_a.get('error') or '')[:300]}")
    L += ["", "## T3a fp16 vs fp32 SSM (long prefixes, target-only, radix off)", "",
          f"- fp16 A/A: {json.dumps(((_E.get('arms') or {}).get('L12_t3b_target') or {}).get('t3a_fp16'), default=str)}",
          f"- fp32 vs fp16: {json.dumps(((_E.get('arms') or {}).get('L15_t3a_fp32') or {}).get('rows'), default=str)[:1500]}",
          "", f"skipped: {json.dumps(_E.get('skipped'))}", "",
          "## Launches", "", "| launch | ok | sec | hook self-test | caps |", "|---|---|---|---|---|"]
    for _n, _r in (_E.get("launches") or {}).items():
        L.append(f"| {_n} | {_r.get('ok')} {_r.get('skipped') or ''} | {_r.get('sec')} | {(_r.get('hook_selftest') or {}).get('ok') if _r.get('hook') else 'off'} | "
                 f"{json.dumps(_r.get('caps'))[:200]} |")
    L += ["", "Verdict ceiling of this session: ENGINE_CORRECTNESS_VALIDATED. No LB uplift is claimed."]
    (E14_OUT / "e17_report.md").write_text("\n".join(L) + "\n", encoding="utf-8")
    print("\n".join(L), flush=True)
    E14_STATE.end("done")
    e14_save()
    _pack = WORKING / "Q27X_E17_BUNDLE.zip"
    with zipfile.ZipFile(_pack, "w", zipfile.ZIP_DEFLATED) as z:
        for fp in sorted(E14_OUT.rglob("*")):
            if fp.is_file() and fp.stat().st_size < 300e6:
                z.write(fp, arcname="e14/" + fp.relative_to(E14_OUT).as_posix())
        for fp in sorted(OUT.glob("sglang_server_*.log")):
            z.write(fp, arcname="logs/" + fp.name)
        for fp in (WORKING / "e14_state.json", WORKING / "e14_heartbeat.log"):
            if fp.exists():
                z.write(fp, arcname=fp.name)
    print("E17 bundle:", _pack, round(_pack.stat().st_size / 1e6, 1), "MB", flush=True)
except BaseException:
    print("E17 report error:", traceback.format_exc()[-2000:], flush=True)
try:
    e11_kill_server(wait=False)
except Exception:
    pass
